# Setup

In [1]:
# Import libs
import spatialdata as sd
from spatialdata_io import xenium
import squidpy as sq
from pathlib import Path
import os, sys
import numpy as np
from scipy.sparse import issparse
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import numpy as np
import pandas as pd
import torch
import glasbey
from tqdm.autonotebook import tqdm
import gc
import time
import os
import copy
import gseapy as gp
import re
from matplotlib.patches import Polygon
from scipy.stats import spearmanr
from scipy.spatial import ConvexHull
from scipy.spatial.distance import pdist
import matplotlib.font_manager as fm
import anndata as ad

from sklearn.decomposition import PCA

from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
import matplotlib.font_manager as fm

from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import logrank_test

import scipy.sparse as sp

from matplotlib.ticker import MultipleLocator
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score



/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/importlib/__init__.py:126: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  return _bootstrap._gcd_import(name[level:], package, level)
/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/xarray_schema/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution
/tmp/ipykernel_1263102/659388111.py:16: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e

In [2]:
# Set seed
sc.settings.verbosity = 3
sc.settings.seed = 0
np.random.seed(0)

In [3]:
# Check if GPU is available
print("GPU Available:", torch.cuda.is_available())

# Check the name of the GPU
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

GPU Available: True
GPU Name: NVIDIA H100 80GB HBM3


In [4]:
# Colours
import colorcet as cc
colors50 = cc.palette["glasbey"][:50]

import random
colors50_spatial = cc.palette["glasbey"][:14]
random.Random(0).shuffle(colors50_spatial)


In [5]:
# Save plot dir
output_dir = '/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Final/Plots/'
sc.settings.figdir = output_dir

In [6]:
# Set scanpy plotting defaults
sc.settings.set_figure_params(
    dpi=300,
    dpi_save=300,
    figsize=(3, 2),
    facecolor='white',
    fontsize=7
)

In [7]:
combined_dir = "/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Subclustering_analysis/combined_for_fig1"

adata_whole = sc.read_h5ad(f"{combined_dir}/adata_whole_combined.h5ad")
adata_T = sc.read_h5ad(f"{combined_dir}/adata_T_combined.h5ad")
adata_TAM = sc.read_h5ad(f"{combined_dir}/adata_TAM_combined.h5ad")

print(f"adata_whole: {adata_whole.shape} obs={list(adata_whole.obs.columns)[:15]}... obsm={list(adata_whole.obsm.keys())}")
print(f"adata_T:     {adata_T.shape}")
print(f"adata_TAM:   {adata_TAM.shape}")


/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


adata_whole: (3092679, 5001) obs=['cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'batch']... obsm=['MintFlow_S_in', 'MintFlow_S_out', 'MintFlow_Xbar_int', 'MintFlow_Xbar_mic', 'MintFlow_Xint', 'MintFlow_Xint (before_sc_pp_normalize_total)', 'MintFlow_Xmic', 'MintFlow_Xmic (before_sc_pp_normalize_total)', 'MintFlow_Z', 'X_PCA_barint', 'X_PCA_barmic', 'X_PCA_int', 'X_PCA_mic', 'X_UMAP_barint', 'X_UMAP_barmic', 'X_UMAP_int', 'X_UMAP_mic', 'X_umap', 'X_umap_Xint', 'X_umap_Xmic', 'X_umap_gex', 'spatial']
adata_T:     (351533, 5001)
adata_TAM:   (373799, 5001)


/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


# Whole dataset analysis

## Xmic embeddings

### Level 2 cell type (Supp) Y

In [8]:
# Xmic UMAP coloured by level-2 cell type (with legend)
sc.pl.embedding(
    adata_whole,
    basis="X_umap_Xmic",
    color="level_2_cell_type",
    show=False,
    frameon=False,
    palette=colors50,
    size=0.01,
    alpha=0.8,
    save="_whole_xmic_umap_level2_celltype.svg",
)
plt.close()


In [9]:
# Xmic UMAP coloured by level-2 cell type (no legend)
sc.pl.embedding(
    adata_whole,
    basis="X_umap_Xmic",
    color="level_2_cell_type",
    show=False,
    frameon=False,
    palette=colors50,
    size=0.01,
    alpha=0.8,
    legend_loc=None,
    save="_whole_xmic_umap_level2_celltype_nolegend.svg",
)
plt.close()


### Batch integration (Supp) Y

In [10]:
# Xmic UMAP coloured by batch
sc.pl.embedding(
    adata_whole,
    basis="X_umap_Xmic",
    color="batch",
    show=False,
    frameon=False,
    palette=colors50,
    size=0.01,
    alpha=0.8,
    legend_loc=None,
    save="_whole_xmic_umap_batch_nolegend.svg",
)
plt.close()


### Clustered Xmic (main) Y

In [11]:
# generated by sc.tl.leiden(adata, resolution=0.50, key_added='leiden_res_0.50', random_state=0)

In [12]:
# Annotate leiden clusters (NaN-safe; already precomputed at build time)
mapping_dict = {
    0: "Tumor epithelial microenvironment",
    1: "Post-treatment tumor epithelial microenvironment",
    2: "Lymphoid-rich infiltrate microenvironment",
    3: "Lymphatic-rich tumor microenvironment",
    4: "Tumor epithelial injury microenvironment",
    5: "Perivascular mural microenvironment",
    6: "Myeloid-rich infiltrate microenvironment",
    7: "Angiogenic remodelling microenvironment",
    8: "Distal tubular microenvironment",
    9: "Microvascular microenvironment 1",
    10: "Proximal tubular microenvironment",
    11: "Microvascular microenvironment 2",
    12: "Lymphoid aggregation microenvironment",
    13: "Contamination",
    14: "Tumor epithelial injury microenvironment",
}

codes = pd.to_numeric(adata_whole.obs["leiden_res_0.50_Xmic"], errors="coerce")
adata_whole.obs["leiden_annots"] = pd.Categorical(codes.map(mapping_dict))


In [13]:
# Xmic UMAP coloured by leiden annotations (with legend)
sc.pl.embedding(
    adata_whole,
    basis="X_umap_Xmic",
    color="leiden_annots",
    show=False,
    frameon=False,
    palette=colors50_spatial,
    size=0.01,
    alpha=0.8,
    save="_whole_xmic_umap_leiden_annots.svg",
)
plt.close()


In [14]:
# Xmic UMAP coloured by leiden annotations (no legend)
sc.pl.embedding(
    adata_whole,
    basis="X_umap_Xmic",
    color="leiden_annots",
    show=False,
    frameon=False,
    palette=colors50_spatial,
    size=0.01,
    alpha=0.8,
    legend_loc=None,
    save="_whole_xmic_umap_leiden_annots_nolegend.svg",
)
plt.close()


In [15]:
# Stacked bar: cell-type composition per leiden cluster
def plot_cluster_composition_stacked_multiple_groups(adata, cluster_col, target_clusters,
                                                     celltype_col="level_3_cell_type", figsize=(10, 8)):
    fig, ax = plt.subplots(figsize=figsize)

    target_clusters = sorted(target_clusters)
    all_celltypes = sorted(adata.obs[celltype_col].dropna().unique())
    celltype_color_dict = {ct: colors50[i % len(colors50)] for i, ct in enumerate(all_celltypes)}
    all_proportions = {}

    for idx, cluster_group in enumerate(target_clusters):
        adata_cluster = adata[adata.obs[cluster_col] == cluster_group].copy()
        celltype_counts = adata_cluster.obs[celltype_col].value_counts()
        proportions = celltype_counts / celltype_counts.sum()
        all_proportions[cluster_group] = proportions

        left = 0
        for ct in all_celltypes:
            if ct in proportions.index:
                value = proportions[ct]
                ax.barh(idx, value, left=left, color=celltype_color_dict[ct])
                left += value

    ax.set_yticks(range(len(target_clusters)))
    ax.set_yticklabels(target_clusters, fontsize=7)
    ax.set_xlabel("Proportion", fontsize=7)
    ax.set_xlim(0, 1)
    ax.invert_yaxis()

    handles = [plt.Rectangle((0, 0), 1, 1, fc=celltype_color_dict[ct]) for ct in all_celltypes]
    ax.legend(handles, all_celltypes,
              title="Cell type",
              bbox_to_anchor=(1.01, 1),
              loc="best",
              fontsize=7,
              frameon=True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

    plt.tight_layout()
    return fig, ax, all_proportions


all_clusters = sorted(adata_whole.obs["leiden_annots"].dropna().unique())

fig, ax, proportions_dict = plot_cluster_composition_stacked_multiple_groups(
    adata_whole,
    cluster_col="leiden_annots",
    target_clusters=all_clusters,
    celltype_col="level_3_cell_type",
    figsize=(6, max(3, 0.25 * len(all_clusters))),
)
fig.savefig(f"{output_dir}/whole_xmic_leiden_composition.svg", bbox_inches="tight")
plt.close(fig)

fig, ax, proportions_dict = plot_cluster_composition_stacked_multiple_groups(
    adata_whole,
    cluster_col="leiden_annots",
    target_clusters=all_clusters,
    celltype_col="level_3_cell_type",
    figsize=(6, max(3, 0.25 * len(all_clusters))),
)
ax.get_legend().remove()
fig.savefig(f"{output_dir}/whole_xmic_leiden_composition_nolegend.svg", bbox_inches="tight")
plt.close(fig)


In [16]:
# Per-section spatial map coloured by leiden annotation
output_dir_spatial = os.path.join(output_dir, "whole_spatial_leiden_annots")
os.makedirs(output_dir_spatial, exist_ok=True)

for batch_id in sorted(adata_whole.obs["batch"].unique()):
    fig, ax = plt.subplots(figsize=(6, 6))

    sc.pl.spatial(
        adata_whole[adata_whole.obs["batch"] == batch_id].copy(),
        color="leiden_annots",
        size=1.5,
        spot_size=20,
        frameon=False,
        show=False,
        ax=ax,
        legend_loc=None,
    )

    scalebar = AnchoredSizeBar(
        ax.transData,
        500,
        "500 µm",
        "lower right",
        pad=0.3,
        color="black",
        frameon=False,
        size_vertical=2,
        fontproperties=fm.FontProperties(size=10),
    )
    ax.add_artist(scalebar)

    raw_name = Path(str(batch_id)).stem
    safe_batch = re.sub(r"[^A-Za-z0-9._-]+", "_", raw_name)

    plt.tight_layout()
    fig.savefig(f"{output_dir_spatial}/{safe_batch}.svg", bbox_inches="tight")
    plt.close(fig)


/tmp/ipykernel_3251087/3651696587.py:8: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/3651696587.py:8: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/3651696587.py:8: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/3651696587.py:8: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/3651696587.py:8: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/3651696587.py:8: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/3651696587.py:8: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/3651696587.py:8: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/3651696587.py:8: FutureWarning: Use `squidpy.pl.spatial_scatter` 

### Cross-embedding Leiden analysis (supp) Y

In [ ]:
# Xmic / Xint / GEX UMAPs coloured by their own leiden solutions
# generated by sc.tl.leiden(adata, resolution=0.50, key_added='leiden_res_0.50', random_state=0)

sc.pl.embedding(
    adata_whole, basis="X_umap_Xmic",
    color="leiden_res_0.50_Xmic",
    show=False, frameon=False, palette=colors50_spatial,
    size=0.01, alpha=0.8, legend_loc="on data",
    save="_whole_xmic_umap_leiden_ondata.svg",
)
plt.close()

sc.pl.embedding(
    adata_whole, basis="X_umap_Xint",
    color="leiden_res_0.50_Xint",
    show=False, frameon=False, palette=colors50_spatial,
    size=0.01, alpha=0.8, legend_loc="on data",
    save="_whole_xint_umap_leiden_ondata.svg",
)
plt.close()

sc.pl.embedding(
    adata_whole, basis="X_umap_gex",
    color="leiden_res_0.50_gex",
    show=False, frameon=False, palette=colors50_spatial,
    size=0.01, alpha=0.8, legend_loc="on data",
    save="_whole_gex_umap_leiden_ondata.svg",
)
plt.close()


In [18]:
xmic_labels = adata_whole.obs["leiden_res_0.50_Xmic"]
xint_labels = adata_whole.obs["leiden_res_0.50_Xint"]
gex_labels = adata_whole.obs["leiden_res_0.50_gex"]

mask = xmic_labels.notna() & xint_labels.notna() & gex_labels.notna()
xmic_labels = xmic_labels[mask]
xint_labels = xint_labels[mask]
gex_labels = gex_labels[mask]

ari_xmic_xint = adjusted_rand_score(xmic_labels, xint_labels)
nmi_xmic_xint = normalized_mutual_info_score(xmic_labels, xint_labels)

ari_xmic_gex = adjusted_rand_score(xmic_labels, gex_labels)
nmi_xmic_gex = normalized_mutual_info_score(xmic_labels, gex_labels)

print(f"Adjusted Rand Index (Xmic vs Xint): {ari_xmic_xint:.4f}")
print(f"Normalized Mutual Information (Xmic vs Xint): {nmi_xmic_xint:.4f}")
print(f"Adjusted Rand Index (Xmic vs GEX):  {ari_xmic_gex:.4f}")
print(f"Normalized Mutual Information (Xmic vs GEX):  {nmi_xmic_gex:.4f}")


Adjusted Rand Index (Xmic vs Xint): 0.2477
Normalized Mutual Information (Xmic vs Xint): 0.4142
Adjusted Rand Index (Xmic vs GEX):  0.4471
Normalized Mutual Information (Xmic vs GEX):  0.5816


## Xint embeddings

### Level 2 cell type (main+supp) Y 

In [19]:
# Xint UMAP coloured by level-2 cell type (with legend)
sc.pl.embedding(
    adata_whole, basis="X_umap_Xint",
    color="level_2_cell_type",
    show=False, frameon=False, palette=colors50,
    size=0.01, alpha=0.8,
    save="_whole_xint_umap_level2_celltype.svg",
)
plt.close()


In [20]:
# Xint UMAP coloured by level-2 cell type (no legend)
sc.pl.embedding(
    adata_whole, basis="X_umap_Xint",
    color="level_2_cell_type",
    show=False, frameon=False, palette=colors50,
    size=0.01, alpha=0.8, legend_loc=None,
    save="_whole_xint_umap_level2_celltype_nolegend.svg",
)
plt.close()


In [21]:
# Xint UMAP coloured by level-2 cell type with tab20 palette (with legend)
sc.pl.embedding(
    adata_whole, basis="X_umap_Xint",
    color="level_2_cell_type",
    show=False, frameon=False, palette="tab20",
    size=0.01, alpha=0.8,
    save="_whole_xint_umap_level2_celltype_tab20.svg",
)
plt.close()


In [22]:
# Xint UMAP coloured by level-2 cell type with tab20 palette (no legend)
sc.pl.embedding(
    adata_whole, basis="X_umap_Xint",
    color="level_2_cell_type",
    show=False, frameon=False, palette="tab20",
    size=0.01, alpha=0.8, legend_loc=None,
    save="_whole_xint_umap_level2_celltype_tab20_nolegend.svg",
)
plt.close()


### Batch (Supp) Y

In [23]:
# Xint UMAP coloured by batch
sc.pl.embedding(
    adata_whole, basis="X_umap_Xint",
    color="batch",
    show=False, frameon=False, palette=colors50,
    size=0.01, alpha=0.8, legend_loc=None,
    save="_whole_xint_umap_batch_nolegend.svg",
)
plt.close()


# Spatial map

### Mintflow signalling score (main) Y

In [24]:
# Per-section spatial map of Mintflow signalling score
output_dir_spatial_signal = os.path.join(output_dir, "whole_spatial_signalling_score")
os.makedirs(output_dir_spatial_signal, exist_ok=True)

batch_str = adata_whole.obs["batch"].astype("string")
unique_batches = sorted(batch_str.dropna().unique().tolist())

for batch_id in unique_batches:
    fig, ax = plt.subplots(figsize=(6, 6))

    mask = batch_str.eq(batch_id).fillna(False).to_numpy(dtype=bool)
    adata_batch = adata_whole[mask].copy()

    vals = adata_batch.obs["signalling_score"].to_numpy()
    vals = vals[np.isfinite(vals)]

    if len(vals) > 0:
        vmin_val = np.percentile(vals, 1)
        vmax_val = np.percentile(vals, 99)
        if vmin_val == vmax_val:
            vmin_val, vmax_val = vals.min(), vals.max()
    else:
        vmin_val, vmax_val = 0, 1

    sc.pl.spatial(
        adata_batch,
        color="signalling_score",
        size=1.5, spot_size=20, frameon=False, show=False, ax=ax,
        legend_loc=None, cmap="rainbow",
        vmin=vmin_val, vmax=vmax_val,
    )

    scalebar = AnchoredSizeBar(
        ax.transData, 500, "500 µm", "lower right",
        pad=0.3, color="black", frameon=False, size_vertical=2,
        fontproperties=fm.FontProperties(size=10),
    )
    ax.add_artist(scalebar)

    raw_name = Path(str(batch_id)).stem
    safe_batch = re.sub(r"[^A-Za-z0-9._-]+", "_", raw_name)
    plt.tight_layout()
    fig.savefig(os.path.join(output_dir_spatial_signal, f"{safe_batch}.svg"), bbox_inches="tight")
    plt.close(fig)


/tmp/ipykernel_3251087/564440636.py:25: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/564440636.py:25: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/564440636.py:25: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/564440636.py:25: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/564440636.py:25: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/564440636.py:25: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/564440636.py:25: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/564440636.py:25: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/564440636.py:25: FutureWarning: Use `squidpy.pl.spatial_scatter` 

### TLS Signature score (supp) Y

In [25]:
# Per-section spatial map of TLS signature score on whole data
TLS_signature = ["CD4", "CD8A", "CD8B", "BCL6", "CXCL13"]

sc.tl.score_genes(
    adata_whole, gene_list=TLS_signature,
    score_name="TLS_Signature_Score", use_raw=False,
)

tls_scores = adata_whole.obs["TLS_Signature_Score"]
adata_whole.obs["TLS_Signature_Score"] = (
    (tls_scores - tls_scores.min()) / (tls_scores.max() - tls_scores.min())
)

scores = adata_whole.obs["TLS_Signature_Score"].astype(float).values
vmin_val = np.percentile(scores, 2.5)
vmax_val = np.percentile(scores, 97.5)

output_dir_spatial_tls = os.path.join(output_dir, "whole_spatial_tls_score")
os.makedirs(output_dir_spatial_tls, exist_ok=True)

for batch_id in sorted(adata_whole.obs["batch"].unique()):
    fig, ax = plt.subplots(figsize=(6, 6))

    adata_batch = adata_whole[adata_whole.obs["batch"] == batch_id].copy()

    sc.pl.spatial(
        adata_batch,
        color="TLS_Signature_Score",
        size=1.5, spot_size=20, frameon=False, show=False, ax=ax,
        legend_loc=None, cmap="rainbow",
        vmin=vmin_val, vmax=vmax_val,
    )

    scalebar = AnchoredSizeBar(
        ax.transData, 500, "500 µm", "lower right",
        pad=0.3, color="black", frameon=False, size_vertical=2,
        fontproperties=fm.FontProperties(size=10),
    )
    ax.add_artist(scalebar)

    raw_name = Path(str(batch_id)).stem
    safe_batch = re.sub(r"[^A-Za-z0-9._-]+", "_", raw_name)
    plt.tight_layout()
    fig.savefig(os.path.join(output_dir_spatial_tls, f"{safe_batch}.svg"), bbox_inches="tight")
    plt.close(fig)


computing score 'TLS_Signature_Score'
    finished: added
    'TLS_Signature_Score', score of gene set (adata.obs).
    247 total control genes are used. (0:00:28)


/tmp/ipykernel_3251087/451214224.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/451214224.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/451214224.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/451214224.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/451214224.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/451214224.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/451214224.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/451214224.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/451214224.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` 

# T cell analysis

In [26]:
adata_T


AnnData object with n_obs × n_vars = 351533 × 5001
    obs: 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'batch', 'collection site type', 'donor', 'Xenium barcode', 'drug', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'outlier', '_scvi_batch', '_scvi_labels', 'SCVI_CLUSTERS_KEY', 'low_quality_cell_highlight', 'highlight', 'level_1_cell_type', 'level_2_cell_type', 'level_3_cell_type', 'cell_status', 'celltype_scanvi', 'C_scANVI', 'section', 'concate_key', 'inflow_CT', 'inflow_BatchID', 'sig_withinColor_noRiboMt_CD8plusT_EM', 'sig_withinColor_noRiboMt_CD8plusT_EMRA', 'sig_withinColor_noRiboMt_CD8plusT_EX_CCL4L2', 'sig_withinColor_noRib

In [27]:
adata_T.obs['Annotation_XMic_Tcells'].value_counts()


Annotation_XMic_Tcells
CD4+ CXCR4hi memory-like T cells          91064
Activated T cells                         78158
LAG3+ IRF1hi IFN-response T cells         71411
NDRG1hi CD8+ T cells                      51581
CD4+ LGMN+ Treg                           35181
LAMP1+ perivascular-associated T cells    24138
Name: count, dtype: int64

In [28]:
adata_T.obs['Annotation_XInt_Tcells'].value_counts()


Annotation_XInt_Tcells
Activated T cells                   224216
NDRG1hi CD8+ T cells                 78187
CD4+ CXCR4hi memory-like T cells     43112
ZNF683+ EM-like T cells               6018
Name: count, dtype: int64

In [29]:
adata_T.obs['Annotation_Tcells_merged'].value_counts()


Annotation_Tcells_merged
CD4+ CXCR4hi memory-like T cells          89312
Activated T cells                         75332
LAG3+ IRF1hi IFN-response T cells         70861
NDRG1hi CD8+ T cells                      51527
CD4+ LGMN+ Treg                           34848
LAMP1+ perivascular-associated T cells    23635
ZNF683+ EM-like T cells                    6018
Name: count, dtype: int64

## Xmic

### Xmic annotation (main) Y

In [ ]:
color_map = {
    "Activated T cells":                        "#d60000",
    "CD4+ CXCR4hi memory-like T cells":         "#8c3bff",
    "CD4+ LGMN+ Treg":                          "#018700",
    "LAG3+ IRF1hi IFN-response T cells":        "#00acc6",
    "LAMP1+ perivascular-associated T cells":   "#97ff00",
    "NDRG1hi CD8+ T cells":                     "#ff7ed1",
}

In [ ]:
key = "Annotation_XMic_Tcells"
cats = adata_T.obs[key].astype("category").cat.categories.tolist()
adata_T.obs[key] = pd.Categorical(adata_T.obs[key], categories=cats, ordered=True)
adata_T.uns[f"{key}_colors"] = [color_map[c] for c in cats]


In [ ]:
# T cells: X_umap_Mic UMAP coloured by Annotation_XMic_Tcells
sc.pl.embedding(
    adata_T, basis="X_umap_Mic",
    color="Annotation_XMic_Tcells",
    show=False, frameon=False,
    size=0.1, alpha=0.8,
    save="_tcell_xmic_umap_annotXMic.svg",
)
plt.close()


In [ ]:
# T cells: X_umap_Mic UMAP coloured by Annotation_XMic_Tcells (no legend)
sc.pl.embedding(
    adata_T, basis="X_umap_Mic",
    color="Annotation_XMic_Tcells",
    show=False, frameon=False,
    size=0.1, alpha=0.8, legend_loc = None,
    save="_tcell_xmic_umap_annotXMic_no_legend.svg",
)
plt.close()


### Xint annotation (supp) Y

In [33]:
# T cells: X_umap_Mic UMAP coloured by Annotation_XInt_Tcells
sc.pl.embedding(
    adata_T, basis="X_umap_Mic",
    color="Annotation_XInt_Tcells",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    save="_tcell_xmic_umap_annotXInt.svg",
)
plt.close()


In [34]:
# T cells: X_umap_Mic UMAP coloured by Annotation_XInt_Tcells (no legend)
sc.pl.embedding(
    adata_T, basis="X_umap_Mic",
    color="Annotation_XInt_Tcells",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    legend_loc=None,
    save="_tcell_xmic_umap_annotXInt_nolegend.svg",
)
plt.close()


### Level 3 cell type (supp) Y

In [35]:
# T cells: X_umap_Mic UMAP coloured by level_3_cell_type
sc.pl.embedding(
    adata_T, basis="X_umap_Mic",
    color="level_3_cell_type",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    save="_tcell_xmic_umap_level3_celltype.svg",
)
plt.close()


In [36]:
# T cells: X_umap_Mic UMAP coloured by level_3_cell_type (no legend)
sc.pl.embedding(
    adata_T, basis="X_umap_Mic",
    color="level_3_cell_type",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    legend_loc=None,
    save="_tcell_xmic_umap_level3_celltype_nolegend.svg",
)
plt.close()


### Merged annotation (supp) Y

In [37]:
# T cells: X_umap_Mic UMAP coloured by Annotation_Tcells_merged
sc.pl.embedding(
    adata_T, basis="X_umap_Mic",
    color="Annotation_Tcells_merged",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    save="_tcell_xmic_umap_annotMerged.svg",
)
plt.close()


In [38]:
# T cells: X_umap_Mic UMAP coloured by Annotation_Tcells_merged (no legend)
sc.pl.embedding(
    adata_T, basis="X_umap_Mic",
    color="Annotation_Tcells_merged",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    legend_loc=None,
    save="_tcell_xmic_umap_annotMerged_nolegend.svg",
)
plt.close()


### Spatial plots of Xmic annotations (main) Y

In [39]:
# Per-section spatial map of T cell Xmic annotations
output_dir_spatial = os.path.join(output_dir, "tcell_spatial_xmic_annot")
os.makedirs(output_dir_spatial, exist_ok=True)

for batch_id in sorted(adata_T.obs["batch"].unique()):
    fig, ax = plt.subplots(figsize=(6, 6))

    subset = adata_T[adata_T.obs["batch"] == batch_id].copy()
    subset.obs[key] = pd.Categorical(subset.obs[key], categories=cats, ordered=True)

    sc.pl.spatial(
        subset,
        color="Annotation_XMic_Tcells",
        size=1.5, spot_size=20, frameon=False, show=False, ax=ax,
        legend_loc=None, palette=color_map,
    )

    scalebar = AnchoredSizeBar(
        ax.transData, 500, "500 µm", "lower right",
        pad=0.3, color="black", frameon=False, size_vertical=2,
        fontproperties=fm.FontProperties(size=10),
    )
    ax.add_artist(scalebar)

    raw_name = Path(str(batch_id)).stem
    safe_batch = re.sub(r"[^A-Za-z0-9._-]+", "_", raw_name)
    plt.tight_layout()
    fig.savefig(f"{output_dir_spatial}/{safe_batch}.svg", bbox_inches="tight")
    plt.close(fig)


/tmp/ipykernel_3251087/3142354063.py:11: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/3142354063.py:11: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/3142354063.py:11: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/3142354063.py:11: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/3142354063.py:11: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/3142354063.py:11: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/3142354063.py:11: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/3142354063.py:11: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/3142354063.py:11: FutureWarning: Use `squidpy.pl.spatial_

In [40]:
# Per-type, per-section spatial maps highlighting one T-cell subtype at a time
output_dir_spatial = os.path.join(output_dir, "tcell_spatial_xmic_annot_by_type")
os.makedirs(output_dir_spatial, exist_ok=True)

annot_col = "Annotation_XMic_Tcells"
adata_T.obs[annot_col] = adata_T.obs[annot_col].astype("category")
cell_types = list(adata_T.obs[annot_col].cat.categories)
color_map = dict(zip(cell_types, colors50[:len(cell_types)]))

for cell_type in cell_types:
    safe_cell_type = re.sub(r"[^A-Za-z0-9._-]+", "_", str(cell_type))
    celltype_dir = os.path.join(output_dir_spatial, safe_cell_type)
    os.makedirs(celltype_dir, exist_ok=True)

    for batch_id in sorted(adata_T.obs["batch"].unique()):
        adata_sub = adata_T[adata_T.obs["batch"] == batch_id].copy()

        adata_sub.obs["highlight_celltype"] = "Other T cells"
        adata_sub.obs.loc[adata_sub.obs[annot_col] == cell_type, "highlight_celltype"] = cell_type

        adata_sub.obs["highlight_celltype"] = pd.Categorical(
            adata_sub.obs["highlight_celltype"],
            categories=["Other T cells", cell_type], ordered=True,
        )

        highlight_palette = {"Other T cells": "lightgray", cell_type: color_map[cell_type]}

        fig, ax = plt.subplots(figsize=(6, 6))
        sc.pl.spatial(
            adata_sub, color="highlight_celltype",
            size=1.5, spot_size=20, frameon=False, show=False, ax=ax,
            legend_loc=None, palette=highlight_palette,
        )

        scalebar = AnchoredSizeBar(
            ax.transData, 500, "500 µm", "lower right",
            pad=0.3, color="black", frameon=False, size_vertical=2,
            fontproperties=fm.FontProperties(size=10),
        )
        ax.add_artist(scalebar)

        raw_name = Path(str(batch_id)).stem
        safe_batch = re.sub(r"[^A-Za-z0-9._-]+", "_", raw_name)
        plt.tight_layout()
        fig.savefig(os.path.join(celltype_dir, f"{safe_batch}_{safe_cell_type}.svg"), bbox_inches="tight")
        plt.close(fig)


/tmp/ipykernel_3251087/1116983790.py:29: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/1116983790.py:29: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/1116983790.py:29: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/1116983790.py:29: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/1116983790.py:29: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/1116983790.py:29: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/1116983790.py:29: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/1116983790.py:29: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/1116983790.py:29: FutureWarning: Use `squidpy.pl.spatial_

In [41]:
# Per-section spatial map of TLS signature score on T cells
TLS_signature = ["CD4", "CD8A", "CD8B", "BCL6", "CXCL13"]

sc.tl.score_genes(
    adata_T, gene_list=TLS_signature,
    score_name="TLS_Signature_Score", use_raw=False,
)

tls_scores = adata_T.obs["TLS_Signature_Score"]
adata_T.obs["TLS_Signature_Score"] = (
    (tls_scores - tls_scores.min()) / (tls_scores.max() - tls_scores.min())
)

scores = adata_T.obs["TLS_Signature_Score"].astype(float).values
vmin_val = np.percentile(scores, 1)
vmax_val = np.percentile(scores, 99)

output_dir_spatial_tls = os.path.join(output_dir, "tcell_spatial_tls_score")
os.makedirs(output_dir_spatial_tls, exist_ok=True)

for batch_id in sorted(adata_T.obs["batch"].unique()):
    fig, ax = plt.subplots(figsize=(6, 6))

    adata_batch = adata_T[adata_T.obs["batch"] == batch_id].copy()

    sc.pl.spatial(
        adata_batch, color="TLS_Signature_Score",
        size=1.5, spot_size=20, frameon=False, show=False, ax=ax,
        legend_loc=None, cmap="rainbow",
        vmin=vmin_val, vmax=vmax_val,
    )

    scalebar = AnchoredSizeBar(
        ax.transData, 500, "500 µm", "lower right",
        pad=0.3, color="black", frameon=False, size_vertical=2,
        fontproperties=fm.FontProperties(size=10),
    )
    ax.add_artist(scalebar)

    raw_name = Path(str(batch_id)).stem
    safe_batch = re.sub(r"[^A-Za-z0-9._-]+", "_", raw_name)
    plt.tight_layout()
    fig.savefig(os.path.join(output_dir_spatial_tls, f"{safe_batch}.svg"), bbox_inches="tight")
    plt.close(fig)


computing score 'TLS_Signature_Score'
    finished: added
    'TLS_Signature_Score', score of gene set (adata.obs).
    199 total control genes are used. (0:00:08)


/tmp/ipykernel_3251087/4211459863.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/4211459863.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/4211459863.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/4211459863.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/4211459863.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/4211459863.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/4211459863.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/4211459863.py:26: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/tmp/ipykernel_3251087/4211459863.py:26: FutureWarning: Use `squidpy.pl.spatial_

### Patient compositions (supp) Y

In [42]:
# Stacked bar: patient composition per TLS-associated T-cell subtype
def plot_cluster_composition_stacked_multiple_groups(adata, cluster_col, target_clusters,
                                                       batch_col="batch", figsize=(10, 8)):
    fig, ax = plt.subplots(figsize=figsize)

    all_batches = adata.obs[batch_col].unique()

    def get_patient_id(batch):
        filename = batch.split("/")[-1] if isinstance(batch, str) else str(batch)
        return filename.split("-")[0]

    all_batches = sorted(all_batches, key=lambda b: get_patient_id(b))
    all_patients = sorted(set(get_patient_id(b) for b in all_batches))
    tab20 = plt.cm.get_cmap("tab20")
    patient_color_dict = {p: tab20(i % 20) for i, p in enumerate(all_patients)}
    batch_color_dict = {b: patient_color_dict[get_patient_id(b)] for b in all_batches}

    all_proportions = {}
    for idx, cluster_group in enumerate(target_clusters):
        adata_cluster = adata[adata.obs[cluster_col] == cluster_group].copy()
        batch_counts = adata_cluster.obs[batch_col].value_counts()
        proportions = batch_counts / batch_counts.sum()
        all_proportions[cluster_group] = proportions

        left = 0
        for batch in all_batches:
            if batch in proportions.index:
                value = proportions[batch]
                ax.barh(idx, value, left=left, color=batch_color_dict[batch])
                left += value

    ax.set_yticks(range(len(target_clusters)))
    ax.set_yticklabels(target_clusters, fontsize=7)
    ax.set_xlabel("Proportion", fontsize=7)
    ax.set_xlim(0, 1)

    handles = [plt.Rectangle((0, 0), 1, 1, fc=patient_color_dict[p]) for p in all_patients]
    ax.legend(handles, all_patients,
              title="Patient", bbox_to_anchor=(1.01, 1),
              loc="best", fontsize=7, frameon=True)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(False)

    plt.tight_layout()
    return fig, ax, all_proportions


tls_clusters = [
    "CD4+ LGMN+ Treg",
    "CD4+ CXCR4hi memory-like T cells",
    "LAG3+ IRF1hi IFN-response T cells",
]

fig, ax, proportions_dict = plot_cluster_composition_stacked_multiple_groups(
    adata_T,
    cluster_col="Annotation_XMic_Tcells",
    target_clusters=tls_clusters,
    batch_col="batch",
    figsize=(6, 3),
)
fig.savefig(f"{output_dir}/tcell_tls_patient_composition.svg", bbox_inches="tight")
plt.close(fig)


/tmp/ipykernel_3251087/627795908.py:14: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  tab20 = plt.cm.get_cmap("tab20")
/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make

## Xint

### Xmic annot (supp) Y

In [43]:
# T cells: X_umap_Int UMAP coloured by Annotation_XMic_Tcells
sc.pl.embedding(
    adata_T, basis="X_umap_Int",
    color="Annotation_XMic_Tcells",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    save="_tcell_xint_umap_annotXMic.svg",
)
plt.close()


In [44]:
# T cells: X_umap_Int UMAP coloured by Annotation_XMic_Tcells (no legend)
sc.pl.embedding(
    adata_T, basis="X_umap_Int",
    color="Annotation_XMic_Tcells",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    legend_loc=None,
    save="_tcell_xint_umap_annotXMic_nolegend.svg",
)
plt.close()


### Xint annot (supp) Y

In [45]:
# T cells: X_umap_Int UMAP coloured by Annotation_XInt_Tcells
sc.pl.embedding(
    adata_T, basis="X_umap_Int",
    color="Annotation_XInt_Tcells",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    save="_tcell_xint_umap_annotXInt.svg",
)
plt.close()


In [46]:
# T cells: X_umap_Int UMAP coloured by Annotation_XInt_Tcells (no legend)
sc.pl.embedding(
    adata_T, basis="X_umap_Int",
    color="Annotation_XInt_Tcells",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    legend_loc=None,
    save="_tcell_xint_umap_annotXInt_nolegend.svg",
)
plt.close()


### Level 3 cell type (supp) Y

In [47]:
# T cells: X_umap_Int UMAP coloured by level_3_cell_type
sc.pl.embedding(
    adata_T, basis="X_umap_Int",
    color="level_3_cell_type",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    save="_tcell_xint_umap_level3_celltype.svg",
)
plt.close()


In [48]:
# T cells: X_umap_Int UMAP coloured by level_3_cell_type (no legend)
sc.pl.embedding(
    adata_T, basis="X_umap_Int",
    color="level_3_cell_type",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    legend_loc=None,
    save="_tcell_xint_umap_level3_celltype_nolegend.svg",
)
plt.close()


### Merged annotation (supp) Y

In [49]:
# T cells: X_umap_Int UMAP coloured by Annotation_Tcells_merged
sc.pl.embedding(
    adata_T, basis="X_umap_Int",
    color="Annotation_Tcells_merged",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    save="_tcell_xint_umap_annotMerged.svg",
)
plt.close()


In [50]:
# T cells: X_umap_Int UMAP coloured by Annotation_Tcells_merged (no legend)
sc.pl.embedding(
    adata_T, basis="X_umap_Int",
    color="Annotation_Tcells_merged",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    legend_loc=None,
    save="_tcell_xint_umap_annotMerged_nolegend.svg",
)
plt.close()


## Gene expression

### Xmic annotation (main) Y

In [51]:
# T cells: X_umap_gex UMAP coloured by Annotation_XMic_Tcells
sc.pl.embedding(
    adata_T, basis="X_umap_gex",
    color="Annotation_XMic_Tcells",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    save="_tcell_gex_umap_annotXMic.svg",
)
plt.close()


In [52]:
# T cells: X_umap_gex UMAP coloured by Annotation_XMic_Tcells (no legend)
sc.pl.embedding(
    adata_T, basis="X_umap_gex",
    color="Annotation_XMic_Tcells",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    legend_loc=None,
    save="_tcell_gex_umap_annotXMic_nolegend.svg",
)
plt.close()


### Gex annotation (supp) Y

In [53]:
# T cells: X_umap_gex UMAP coloured by Annotation_GEX_Tcells
sc.pl.embedding(
    adata_T, basis="X_umap_gex",
    color="Annotation_GEX_Tcells",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    save="_tcell_gex_umap_annotGEX.svg",
)
plt.close()


In [54]:
# T cells: X_umap_gex UMAP coloured by Annotation_XMic_Tcells (no legend)
sc.pl.embedding(
    adata_T, basis="X_umap_gex",
    color="Annotation_XMic_Tcells",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    legend_loc=None,
    save="_tcell_gex_umap_annotXMic_nolegend_v2.svg",
)
plt.close()


### Merged annotation (supp) Y

In [55]:
# T cells: X_umap_gex UMAP coloured by Annotation_Tcells_merged
sc.pl.embedding(
    adata_T, basis="X_umap_gex",
    color="Annotation_Tcells_merged",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    save="_tcell_gex_umap_annotMerged.svg",
)
plt.close()


In [56]:
# T cells: X_umap_gex UMAP coloured by Annotation_Tcells_merged (no legend)
sc.pl.embedding(
    adata_T, basis="X_umap_gex",
    color="Annotation_Tcells_merged",
    show=False, frameon=False, palette=colors50,
    size=0.1, alpha=0.8,
    legend_loc=None,
    save="_tcell_gex_umap_annotMerged_nolegend.svg",
)
plt.close()


## Misc

### scRNAseq gene scoring (main)

In [57]:
# Load adata
adata = sc.read_h5ad("/lustre/scratch126/cellgen/lotfollahi/zh4/Mintflow/Validation_scRNAseq/RCC_upload_final_raw_counts.h5ad 2")


In [58]:
adata

AnnData object with n_obs × n_vars = 270855 × 19736
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'patient', 'percent.mt', 'summaryDescription', 'annotation', 'region', 'broad_type'
    var: 'name'
    uns: 'annotation_colors', 'broad_type_colors', 'region_colors', 'summaryDescription_colors'
    obsm: 'X_pca', 'X_umap'

In [59]:
## Tidy up to only include RCC tumour samples
adata = adata[~adata.obs['patient'].isin(['PD44714', 'PD47172'])].copy() 
adata = adata[adata.obs['summaryDescription'].isin(['Tumour'])].copy()


In [60]:
gene_set1 = ['CD4', 'CD14', 'LGMN', 'GRN', 'SLCO2B1', 'CD68', 'STAB1', 'MSR1', 'CD163', 'CSF1R', 'MAN2B1', 'GAA', 'SLC40A1', 'ITGB2', 'CYBA', 'CYBB', 'SIGLEC1', 'AP1B1', 'CTSL', 'AXL', 'CPVL', 'CTSC', 'SDC3', 'FCGR3A', 'LIPA', 'NCOA4', 'CREG1', 'MAFB', 'NCF1', 'HCK', 'SPI1', 'FUCA1', 'CIITA', 'SCPEP1', 'TBXAS1', 'MMP14', 'CTSH', 'FCGR2A', 'LGALS9', 'LAIR1', 'THEMIS2', 'STAT1', 'F13A1', 'GAS6', 'MRC2', 'TRPM2', 'VSIG4', 'HMOX1', 'ABCA1', 'CD300A', 'SLC7A7', 'ITGAX', 'LRP1', 'FERMT3', 'SAMHD1', 'TYMP', 'C3AR1', 'TREM2', 'PLTP', 'CMKLR1', 'LILRB1', 'OLFML3', 'CTSA', 'NAIP', 'SGK1', 'GLB1', 'UCP2', 'ENG', 'WAS', 'FCGR1A', 'CXCL16', 'GNAI2', 'FOLR2', 'PLAUR', 'AOAH', 'TNFSF13', 'ADGRE2', 'ST14', 'LAMP1', 'ABI3', 'PFKFB3', 'CXCL12', 'IFNGR1', 'MRC1', 'ZYX', 'GPR34', 'WARS', 'CCR1', 'SCIMP', 'ACP5', 'PLEKHO1', 'CEBPA', 'ARRB2', 'CSF3R', 'GIMAP5', 'AKR1B1', 'NR1H3', 'TKT', 'NINJ1', 'PTAFR']

gene_set2 = ['CXCR4', 'MYH9', 'SMAP2', 'CD3E', 'CD44', 'ERN1', 'SELPLG', 'EEF1G', 'LENG8', 'TMEM173', 'ITK', 'ZAP70', 'CD5', 'FYN', 'SEMA4D', 'KCNA3', 'PRDM1', 'ACAP1', 'IKZF3', 'FCMR', 'STAT4', 'CD4', 'TCF7', 'IKZF1', 'CD6', 'PDCD4', 'DDX39B', 'OGT', 'SEPTIN9', 'STK4', 'PTK2B', 'PBXIP1', 'TC2N', 'GIMAP5', 'CD40LG', 'NLRP1', 'SEPTIN6', 'ITGAL', 'PLEC', 'CD247', 'GRK2', 'PLCG1', 'SH3BP5', 'CTBP1', 'TGFBR2', 'PRKCQ', 'IL16', 'CDC14A', 'PDE4B', 'ETS1', 'FLT3LG', 'CCR7', 'CD96', 'HNRNPH1', 'RUNX2', 'KLRG1', 'SMAD3', 'SIGIRR', 'LY9', 'EPHA1', 'CD99', 'DIP2A', 'PTGER4', 'FBLN5', 'CCN2', 'ECRG4', 'TNK2', 'HECA', 'TSPAN14', 'TNFSF8', 'CCR6', 'MAP4K2', 'SORL1', 'CCR4', 'SLAMF1', 'LEF1', 'ADGRE5', 'IL11RA', 'KDM5D', 'SYNPO2', 'SLC2A3', 'S1PR1', 'SELL', 'ABCA7', 'AEBP1', 'CCL19', 'FOXP1', 'CD28', 'NIBAN1', 'CASP8', 'CXCR6', 'ATM', 'SATB1', 'CCDC80', 'KLRB1', 'SP140', 'EPHA4', 'SLAMF6', 'SFRP1', 'CFLAR']

gene_set3 = ['CD27', 'CST7', 'LAG3', 'IRF1', 'ITGAL', 'CD8A', 'TAP1', 'IL2RG', 'TNFSF10', 'STAT1', 'TNFRSF9', 'SPN', 'CXCR3', 'PTPN6', 'TAP2', 'ITGB2', 'CSF1', 'CD3E', 'GZMA', 'NLRC5', 'RASGRP1', 'GZMK', 'NTNG2', 'GBP5', 'PIK3CD', 'GBP1', 'JAK3', 'HNRNPLL', 'IL2RB', 'IGF2R', 'APOBEC3G', 'GPR174', 'CD2', 'TRAF5', 'ITGA4', 'SEMA4D', 'CDH6', 'ITGB7', 'CD247', 'CD8B', 'CSK', 'TOX2', 'GRK2', 'PRKCH', 'SIRPG', 'APMAP', 'APOBEC3C', 'SH2D2A', 'IL16', 'LAT', 'IL10RA', 'VEGFA', 'LBH', 'HDAC1', 'CD3G', 'TNFRSF14', 'IKZF3', 'CD82', 'CPT1A', 'CHD3', 'CP', 'TOX', 'FASLG', 'ADAR', 'RASSF5', 'C4B', 'PDIA3', 'BTN3A1', 'FCRL3', 'MCTP2', 'CCR5', 'EOMES', 'UBD', 'OGT', 'DPP4', 'KLRK1', 'TNFRSF1B', 'ATXN1', 'LCK', 'PECAM1', 'UBASH3A', 'KMT2A', 'CIITA', 'NCOR2', 'UCP2', 'WARS', 'CD38', 'HNRNPH1', 'HNRNPD', 'RASSF1', 'ADGRE5', 'IL21R', 'GTF2I', 'ITM2A', 'CD96', 'DNM2', 'HAVCR2', 'ATXN2L', 'PRF1', 'GUSB']

t_cell_value = 'T-cell' 

is_t_cell = adata.obs['broad_type'] == t_cell_value
adata_t = adata[is_t_cell].copy()

# Calculate scores only on T cells
sc.tl.score_genes(adata_t, gene_list=gene_set1, score_name='CD4+ LGMN+ Treg') 
sc.tl.score_genes(adata_t, gene_list=gene_set2, score_name='CD4+ CXCR4hi memory-like T cells')
sc.tl.score_genes(adata_t, gene_list=gene_set3, score_name='LAG3+ IRF1hi IFN-response T cells')

# Initialize scores in full adata as NaN
for score in ['CD4+ LGMN+ Treg', 'CD4+ CXCR4hi memory-like T cells', 'LAG3+ IRF1hi IFN-response T cells']:
    adata.obs[score] = np.nan

# Assign scores back to T cells in full adata
for score in ['CD4+ LGMN+ Treg', 'CD4+ CXCR4hi memory-like T cells', 'LAG3+ IRF1hi IFN-response T cells']:
    adata.obs.loc[is_t_cell, score] = adata_t.obs[score]




computing score 'CD4+ LGMN+ Treg'
    finished: added
    'CD4+ LGMN+ Treg', score of gene set (adata.obs).
    947 total control genes are used. (0:00:47)
computing score 'CD4+ CXCR4hi memory-like T cells'
    finished: added
    'CD4+ CXCR4hi memory-like T cells', score of gene set (adata.obs).
    748 total control genes are used. (0:00:46)
computing score 'LAG3+ IRF1hi IFN-response T cells'
    finished: added
    'LAG3+ IRF1hi IFN-response T cells', score of gene set (adata.obs).
    790 total control genes are used. (0:00:43)


In [ ]:
# Overlay T-cell module scores on scRNAseq UMAP (T cells only, rest in grey)
module_scores = [
    "CD4+ LGMN+ Treg",
    "CD4+ CXCR4hi memory-like T cells",
    "LAG3+ IRF1hi IFN-response T cells",
]

percentile_ranges = {
    "CD4+ LGMN+ Treg": (0.1, 99.9),
    "CD4+ CXCR4hi memory-like T cells": (0.1, 99.9),
    "LAG3+ IRF1hi IFN-response T cells": (0.1, 99.9),
}

for score in module_scores:
    original_values = adata.obs[score].to_numpy()
    low_q, high_q = percentile_ranges.get(score, (0.1, 99.9))
    vmin, vmax = np.nanpercentile(original_values, [low_q, high_q])

    masked_score = original_values.copy()
    masked_score[~is_t_cell] = np.nan
    temp_col = f"{score}_masked_t_cells"
    adata.obs[temp_col] = masked_score

    safe_score = re.sub(r"[^A-Za-z0-9._-]+", "_", score)
    sc.pl.umap(
        adata,
        color=temp_col,
        show=False,
        frameon=False,
        cmap="rainbow",
        na_color="lightgrey",
        vmin=vmin,
        vmax=vmax,
        title=f"{score} Signature (T cells only)",
        size=1,
        alpha=0.8,
        save=f"_scrnaseq_signature_{safe_score}_tcells.svg",
    )
    plt.close()

    if temp_col in adata.obs.columns:
        del adata.obs[temp_col]


In [62]:
# Combined T-cell state annotations UMAP
adata.obs["combined_tcell_states"] = np.select(
    [
        adata.obs["annotation"].eq("CD4+Treg"),
        adata.obs["annotation"].isin(["CD4+T_Act-CXCR4"]),
        adata.obs["annotation"].isin(["CD8+T_EX-IL10"]),
        adata.obs["annotation"].isin(["CD8+T_preEX-PMCH"]),
    ],
    [
        "CD4+Treg",
        "CD4+T_Act-CXCR4",
        "CD8+T_EX-IL10",
        "CD8+T_preEX-PMCH",
    ],
    default="Other",
)

adata.obs["combined_tcell_states"] = pd.Categorical(
    adata.obs["combined_tcell_states"],
    categories=[
        "CD4+Treg",
        "CD4+T_Act-CXCR4",
        "CD8+T_EX-IL10",
        "CD8+T_preEX-PMCH",
        "Other",
    ],
)

sc.pl.umap(
    adata,
    color="combined_tcell_states",
    show=False,
    frameon=False,
    palette={
        "CD4+Treg": "#2ca02c",
        "CD4+T_Act-CXCR4": "#ff7f0e",
        "CD8+T_EX-IL10": "#d62728",
        "CD8+T_preEX-PMCH": "#1f77b4",
        "Other": "lightgrey",
    },
    title="Combined T-cell signatures",
    size=5,
    alpha=0.8,
    groups=[
        "CD4+Treg",
        "CD4+T_Act-CXCR4",
        "CD8+T_EX-IL10",
        "CD8+T_preEX-PMCH",
    ],
    save="_scrnaseq_combined_tcell_states.svg",
)
plt.close()


### Forest plots of TLS (supp) Y

In [8]:
# Define the batches to analyze
TLS_validated_batches = [
    '/nfs/team361/dj17/MintFlow_2025/xenium_RCC/annotated_data/CV1-KID-0-FO-1.h5ad',
    '/nfs/team361/dj17/MintFlow_2025/xenium_RCC/annotated_data/CV1-KID-0-FT-2.h5ad',
    '/nfs/team361/dj17/MintFlow_2025/xenium_RCC/unannotated_data/CV5-KID-0-FO-1.h5ad',
    '/nfs/team361/dj17/MintFlow_2025/xenium_RCC/unannotated_data/CV7-KID-0-FT-2-s3.h5ad',
    '/nfs/team361/dj17/MintFlow_2025/xenium_RCC/annotated_data/CV9-KID-0-FT-2.h5ad',
    '/nfs/team361/dj17/MintFlow_2025/xenium_RCC/annotated_data/DI13-KID-0-FT-1.h5ad',
    '/nfs/team361/dj17/MintFlow_2025/xenium_RCC/annotated_data/CV6-KID-0-FT-1.h5ad',
    '/nfs/team361/dj17/MintFlow_2025/xenium_RCC/annotated_data/DI10-KID-0-FT-3.h5ad',
]

# Calculate TLS signature score
TLS_signature = ['CD4', 'CD8A', 'CD8B', 'BCL6', 'CXCL13']
sc.tl.score_genes(
    adata_T,
    gene_list=TLS_signature,
    score_name='TLS_Signature_Score',
    use_raw=False
)

tls_scores = adata_T.obs['TLS_Signature_Score']
tls_scores_normalized = (tls_scores - tls_scores.min()) / (tls_scores.max() - tls_scores.min())
adata_T.obs['TLS_Signature_Score_Normalized'] = tls_scores_normalized

# Define top DE genes per cluster group
cluster_top_genes = {
    'CD4+ LGMN+ Treg': ['CD4', 'CD14', 'LGMN', 'GRN', 'SLCO2B1', 'CD68', 'STAB1', 'MSR1', 'CD163', 'CSF1R', 'MAN2B1', 'GAA', 'SLC40A1', 'ITGB2', 'CYBA', 'CYBB', 'SIGLEC1', 'AP1B1', 'CTSL', 'AXL', 'CPVL', 'CTSC', 'SDC3', 'FCGR3A', 'LIPA', 'NCOA4', 'CREG1', 'MAFB', 'NCF1', 'HCK', 'SPI1', 'FUCA1', 'CIITA', 'SCPEP1', 'TBXAS1', 'MMP14', 'CTSH', 'FCGR2A', 'LGALS9', 'LAIR1', 'THEMIS2', 'STAT1', 'F13A1', 'GAS6', 'MRC2', 'TRPM2', 'VSIG4', 'HMOX1', 'ABCA1', 'CD300A', 'SLC7A7', 'ITGAX', 'LRP1', 'FERMT3', 'SAMHD1', 'TYMP', 'C3AR1', 'TREM2', 'PLTP', 'CMKLR1', 'LILRB1', 'OLFML3', 'CTSA', 'NAIP', 'SGK1', 'GLB1', 'UCP2', 'ENG', 'WAS', 'FCGR1A', 'CXCL16', 'GNAI2', 'FOLR2', 'PLAUR', 'AOAH', 'TNFSF13', 'ADGRE2', 'ST14', 'LAMP1', 'ABI3', 'PFKFB3', 'CXCL12', 'IFNGR1', 'MRC1', 'ZYX', 'GPR34', 'WARS', 'CCR1', 'SCIMP', 'ACP5', 'PLEKHO1', 'CEBPA', 'ARRB2', 'CSF3R', 'GIMAP5', 'AKR1B1', 'NR1H3', 'TKT', 'NINJ1', 'PTAFR'],
    'CD4+ CXCR4hi memory-like T cells': ['CXCR4', 'MYH9', 'SMAP2', 'CD3E', 'CD44', 'ERN1', 'SELPLG', 'EEF1G', 'LENG8', 'TMEM173', 'ITK', 'ZAP70', 'CD5', 'FYN', 'SEMA4D', 'KCNA3', 'PRDM1', 'ACAP1', 'IKZF3', 'FCMR', 'STAT4', 'CD4', 'TCF7', 'IKZF1', 'CD6', 'PDCD4', 'DDX39B', 'OGT', 'SEPTIN9', 'STK4', 'PTK2B', 'PBXIP1', 'TC2N', 'GIMAP5', 'CD40LG', 'NLRP1', 'SEPTIN6', 'ITGAL', 'PLEC', 'CD247', 'GRK2', 'PLCG1', 'SH3BP5', 'CTBP1', 'TGFBR2', 'PRKCQ', 'IL16', 'CDC14A', 'PDE4B', 'ETS1', 'FLT3LG', 'CCR7', 'CD96', 'HNRNPH1', 'RUNX2', 'KLRG1', 'SMAD3', 'SIGIRR', 'LY9', 'EPHA1', 'CD99', 'DIP2A', 'PTGER4', 'FBLN5', 'CCN2', 'ECRG4', 'TNK2', 'HECA', 'TSPAN14', 'TNFSF8', 'CCR6', 'MAP4K2', 'SORL1', 'CCR4', 'SLAMF1', 'LEF1', 'ADGRE5', 'IL11RA', 'KDM5D', 'SYNPO2', 'SLC2A3', 'S1PR1', 'SELL', 'ABCA7', 'AEBP1', 'CCL19', 'FOXP1', 'CD28', 'NIBAN1', 'CASP8', 'CXCR6', 'ATM', 'SATB1', 'CCDC80', 'KLRB1', 'SP140', 'EPHA4', 'SLAMF6', 'SFRP1', 'CFLAR'],
    'LAG3+ IRF1hi IFN-response T cells': ['CD27', 'CST7', 'LAG3', 'IRF1', 'ITGAL', 'CD8A', 'TAP1', 'IL2RG', 'TNFSF10', 'STAT1', 'TNFRSF9', 'SPN', 'CXCR3', 'PTPN6', 'TAP2', 'ITGB2', 'CSF1', 'CD3E', 'GZMA', 'NLRC5', 'RASGRP1', 'GZMK', 'NTNG2', 'GBP5', 'PIK3CD', 'GBP1', 'JAK3', 'HNRNPLL', 'IL2RB', 'IGF2R', 'APOBEC3G', 'GPR174', 'CD2', 'TRAF5', 'ITGA4', 'SEMA4D', 'CDH6', 'ITGB7', 'CD247', 'CD8B', 'CSK', 'TOX2', 'GRK2', 'PRKCH', 'SIRPG', 'APMAP', 'APOBEC3C', 'SH2D2A', 'IL16', 'LAT', 'IL10RA', 'VEGFA', 'LBH', 'HDAC1', 'CD3G', 'TNFRSF14', 'IKZF3', 'CD82', 'CPT1A', 'CHD3', 'CP', 'TOX', 'FASLG', 'ADAR', 'RASSF5', 'C4B', 'PDIA3', 'BTN3A1', 'FCRL3', 'MCTP2', 'CCR5', 'EOMES', 'UBD', 'OGT', 'DPP4', 'KLRK1', 'TNFRSF1B', 'ATXN1', 'LCK', 'PECAM1', 'UBASH3A', 'KMT2A', 'CIITA', 'NCOR2', 'UCP2', 'WARS', 'CD38', 'HNRNPH1', 'HNRNPD', 'RASSF1', 'ADGRE5', 'IL21R', 'GTF2I', 'ITM2A', 'CD96', 'DNM2', 'HAVCR2', 'ATXN2L', 'PRF1', 'GUSB'],
}

batch_mask = adata_T.obs['batch'].isin(TLS_validated_batches)
adata_filtered = adata_T[batch_mask].copy()

available_genes = adata_filtered.var_names.tolist()

for cluster_name, gene_list in cluster_top_genes.items():
    genes_present = [g for g in gene_list if g in available_genes]
    score_col = f'{cluster_name}_DE_score'
    sc.tl.score_genes(adata_filtered, gene_list=genes_present, score_name=score_col, use_raw=False)

cluster_score_cols = list(cluster_top_genes.keys())
unique_batches = sorted(adata_filtered.obs['batch'].unique())

# Bootstrap CI
B = 1000
np.random.seed(42)

def bootstrap_spearman(x, y, B=1000, seed=42):
    rng = np.random.RandomState(seed)
    n = len(x)
    rho_obs, pval = spearmanr(x, y)
    rho_boot = np.empty(B)
    for b in range(B):
        idx = rng.choice(n, size=n, replace=True)
        rho_boot[b], _ = spearmanr(x[idx], y[idx])
    ci_lo = np.nanpercentile(rho_boot, 2.5)
    ci_hi = np.nanpercentile(rho_boot, 97.5)
    return rho_obs, pval, ci_lo, ci_hi

all_results = []

for batch_id in tqdm(unique_batches, desc='Computing Spearman per section'):
    adata_batch = adata_filtered[adata_filtered.obs['batch'] == batch_id]
    batch_label = os.path.basename(batch_id).replace('.h5ad', '')
    tls = adata_batch.obs['TLS_Signature_Score_Normalized'].astype(float).values
    n_cells = len(tls)

    for cluster_name in cluster_score_cols:
        score_col = f'{cluster_name}_DE_score'
        y = adata_batch.obs[score_col].astype(float).values

        if n_cells > 10:
            rho_obs, pval, ci_lo, ci_hi = bootstrap_spearman(tls, y, B=B, seed=42)
        else:
            rho_obs, pval, ci_lo, ci_hi = np.nan, np.nan, np.nan, np.nan

        all_results.append({
            'section': batch_label,
            'cluster': cluster_name,
            'rho': rho_obs,
            'pval': pval,
            'ci_lo': ci_lo,
            'ci_hi': ci_hi,
            'n_cells': n_cells,
        })

results_df = pd.DataFrame(all_results)

# Forest plots of per-section TLS-vs-signature Spearman correlations
cluster_colors_forest = {
    "CD4+ LGMN+ Treg": "#377EB8",
    "CD4+ CXCR4hi memory-like T cells": "#E41A1C",
    "LAG3+ IRF1hi IFN-response T cells": "#CA0EEB",
}

for cluster_name in cluster_score_cols:
    color = cluster_colors_forest[cluster_name]

    sub = results_df[results_df["cluster"] == cluster_name].copy()
    sub = sub.sort_values("section")

    sections = sub["section"].tolist()
    rhos = sub["rho"].values
    avg_rho = np.nanmean(rhos)
    ci_los = sub["ci_lo"].values
    ci_his = sub["ci_hi"].values
    y_positions = np.arange(len(sections))

    fig, ax = plt.subplots(figsize=(3, 2))

    xerr_low = rhos - ci_los
    xerr_high = ci_his - rhos

    ax.errorbar(
        rhos, y_positions,
        xerr=np.array([xerr_low, xerr_high]),
        fmt="none", ecolor="#000000", elinewidth=1, capsize=0, zorder=2,
    )

    ax.scatter(rhos, y_positions, s=5, color=color, linewidth=0.7, zorder=3)

    ax.axvline(x=0, color="grey", linewidth=1.5, zorder=1, linestyle="--")
    ax.axvline(x=avg_rho, color=color, linewidth=1, zorder=1, linestyle=":")

    ax.set_yticks(y_positions)
    ax.set_yticklabels(sections, fontsize=7)
    ax.set_xlabel("Spearman ρ", fontsize=7)
    ax.set_title(cluster_name, fontsize=7)
    ax.invert_yaxis()
    ax.set_xlim(-0.2, 1.0)
    ax.grid(False)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    safe_cluster = re.sub(r"[^A-Za-z0-9._-]+", "_", cluster_name)
    plt.tight_layout()
    fig.savefig(
        os.path.join(output_dir, f"tcell_forest_tls_spearman_{safe_cluster}.svg"),
        bbox_inches="tight",
    )
    plt.close(fig)

computing score 'TLS_Signature_Score'
    finished: added
    'TLS_Signature_Score', score of gene set (adata.obs).
    199 total control genes are used. (0:00:01)
computing score 'CD4+ LGMN+ Treg_DE_score'


/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/anndata/_core/anndata.py:1796: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


    finished: added
    'CD4+ LGMN+ Treg_DE_score', score of gene set (adata.obs).
    578 total control genes are used. (0:00:00)
computing score 'CD4+ CXCR4hi memory-like T cells_DE_score'
    finished: added
    'CD4+ CXCR4hi memory-like T cells_DE_score', score of gene set (adata.obs).
    631 total control genes are used. (0:00:00)
computing score 'LAG3+ IRF1hi IFN-response T cells_DE_score'
    finished: added
    'LAG3+ IRF1hi IFN-response T cells_DE_score', score of gene set (adata.obs).
    285 total control genes are used. (0:00:00)


Computing Spearman per section:   0%|          | 0/8 [00:00<?, ?it/s]

### Colorectal cancer validation (supp) Y

In [8]:
# CRC validation: per-patient TLS signature scores on spatial plots
tls_signatures = {
    "TLS_crc": [
        "ITGAM", "MS4A1", "CD3D", "CD79A", "CR2", "FCER2",
        "CXCR5", "NTAN1", "CCL21", "CXCL13", "LYVE1",
    ],
    "TLS_general_cxcr4_cd4_t_cells": [
        "CXCR4", "MYH9", "SMAP2", "CD3E", "CD44", "ERN1", "SELPLG", "EEF1G", "LENG8",
        "TMEM173", "ITK", "ZAP70", "CD5", "FYN", "SEMA4D", "KCNA3", "PRDM1", "ACAP1",
        "IKZF3", "FCMR", "STAT4", "CD4", "TCF7", "IKZF1", "CD6", "PDCD4", "DDX39B", "OGT",
        "SEPTIN9", "STK4", "PTK2B", "PBXIP1", "TC2N", "GIMAP5", "CD40LG", "NLRP1",
        "SEPTIN6", "ITGAL", "PLEC", "CD247", "GRK2", "PLCG1", "SH3BP5", "CTBP1", "TGFBR2",
        "PRKCQ", "IL16", "CDC14A", "PDE4B", "ETS1",
    ],
}

patients = {
    "P1": {
        "path": "/nfs/team361/ms83/data/CRC/visiumHD_mintflow/raw_data/adata_sc_P1_rawformintflow_7K_svg.h5ad",
        "vmax": 0.15,
    },
    "P2": {
        "path": "/nfs/team361/ms83/data/CRC/visiumHD_mintflow/raw_data/adata_sc_P2_rawformintflow_7K_svg.h5ad",
        "vmax": 0.4,
    },
    "P5": {
        "path": "/nfs/team361/ms83/data/CRC/visiumHD_mintflow/Outputs_nb1_P5_annealing/adata_P5_mic_score.h5ad",
        "vmax": 0.2,
    },
}

output_dir_crc = os.path.join(output_dir, "crc_validation")
os.makedirs(output_dir_crc, exist_ok=True)

for patient_id, cfg in patients.items():
    print(f"\n--- {patient_id} ---")
    adata = sc.read_h5ad(cfg["path"])

    score_ranges = {}
    for score_name, gene_list in tls_signatures.items():
        valid_genes = [g for g in gene_list if g in adata.var_names]
        if not valid_genes:
            print(f"Skipping {score_name}: no genes found")
            continue

        sc.tl.score_genes(adata, valid_genes, score_name=score_name)
        score_values = adata.obs[score_name].astype(float).to_numpy()
        score_ranges[score_name] = {
            "n_genes": len(valid_genes),
            "min": float(np.nanmin(score_values)),
            "max": float(np.nanmax(score_values)),
        }

    print(f"Loaded {patient_id}: {adata.n_obs} spots x {adata.n_vars} genes")
    print(pd.DataFrame(score_ranges).T)

    if "DeconvolutionLabel1" in adata.obs.columns:
        fig = sq.pl.spatial_scatter(
            adata,
            library_id="spatial",
            shape=None,
            color=["DeconvolutionLabel1"],
            wspace=0.4,
            size=0.02,
            figsize=(15, 15),
            frameon=False,
            return_ax=True,
        ).figure
        fig.savefig(
            os.path.join(output_dir_crc, f"crc_{patient_id}_deconvlabel.svg"),
            bbox_inches="tight",
        )
        plt.close(fig)
    else:
        print("DeconvolutionLabel1 not found in adata.obs")

    for score_name in tls_signatures:
        if score_name not in adata.obs.columns:
            print(f"{score_name} not found in adata.obs")
            continue

        fig = sq.pl.spatial_scatter(
            adata,
            library_id="spatial",
            shape=None,
            color=[score_name],
            wspace=0.4,
            size=0.02,
            figsize=(6, 6),
            frameon=False,
            vmin=0,
            vmax=cfg["vmax"],
            return_ax=True,
        ).figure
        fig.savefig(
            os.path.join(output_dir_crc, f"crc_{patient_id}_{score_name}.svg"),
            bbox_inches="tight",
        )
        plt.close(fig)



--- P1 ---
computing score 'TLS_crc'
    finished: added
    'TLS_crc', score of gene set (adata.obs).
    298 total control genes are used. (0:00:00)
computing score 'TLS_general_cxcr4_cd4_t_cells'
    finished: added
    'TLS_general_cxcr4_cd4_t_cells', score of gene set (adata.obs).
    891 total control genes are used. (0:00:00)
Loaded P1: 235530 spots x 7000 genes
                               n_genes       min       max
TLS_crc                            8.0 -0.371644  1.469799
TLS_general_cxcr4_cd4_t_cells     46.0 -0.253233  0.339677


/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/squidpy/pl/_spatial_utils.py:982: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  _cax = scatter(



--- P2 ---
computing score 'TLS_crc'
    finished: added
    'TLS_crc', score of gene set (adata.obs).
    149 total control genes are used. (0:00:01)
computing score 'TLS_general_cxcr4_cd4_t_cells'
    finished: added
    'TLS_general_cxcr4_cd4_t_cells', score of gene set (adata.obs).
    793 total control genes are used. (0:00:01)
Loaded P2: 322327 spots x 7000 genes
                               n_genes       min       max
TLS_crc                            6.0 -0.208054  1.639821
TLS_general_cxcr4_cd4_t_cells     37.0 -0.414335  0.671927


/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/squidpy/pl/_spatial_utils.py:982: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  _cax = scatter(



--- P5 ---
computing score 'TLS_crc'
    finished: added
    'TLS_crc', score of gene set (adata.obs).
    349 total control genes are used. (0:00:00)
computing score 'TLS_general_cxcr4_cd4_t_cells'
    finished: added
    'TLS_general_cxcr4_cd4_t_cells', score of gene set (adata.obs).
    887 total control genes are used. (0:00:01)
Loaded P5: 252242 spots x 7000 genes
                               n_genes       min       max
TLS_crc                           10.0 -0.134670  1.391404
TLS_general_cxcr4_cd4_t_cells     43.0 -0.373719  0.562938


/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/squidpy/pl/_spatial_utils.py:982: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap', 'norm' will be ignored
  _cax = scatter(


### Colorectal mouse validation (supp) Y

In [9]:
adata_cd8 = sc.read_h5ad("/lustre/scratch126/cellgen/clatworthy/cl28/share/Daniyal/Kaede/processed_CD8T_tumour.h5ad")

# Create a new column in adata.obs
adata_cd8.obs['photoconversion_status'] = adata_cd8.obs['condition'].replace({
    'Green_Isotype': 'green',
    'Green_anti-PDL1': 'green',
    'Red_Isotype': 'red',
    'Red_anti-PDL1': 'red'
})

print(adata_cd8)

AnnData object with n_obs × n_vars = 16145 × 18399
    obs: 'sample', 'treatment', 'colour', 'sort_group', 'scrublet_score', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'gmm_pct_count_clusters_keep', 'is_doublet', 'filter_rna', 'batch', 'S_score', 'G2M_score', 'phase', 'condition', 'seq_batch', 'celltype_cluster_fine', 'celltype_fine_latest', 'photoconversion_status'
    var: 'gene_ids', 'feature_types', 'genome', 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    obsm: 'X_draw_graph_fa', 'X_pca', 'X_umap'
    obsp: 'connectivities', 'distances'


/tmp/ipykernel_1191634/3701251783.py:4: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  adata_cd8.obs['photoconversion_status'] = adata_cd8.obs['condition'].replace({


In [10]:
adata_cd8.obs['celltype_coarse'] = (
    adata_cd8.obs['celltype_fine_latest']
    .str.replace(r'\.\d+$', '', regex=True)
)

In [11]:
# Mouse CD8 UMAPs coloured by cell type, treatment, and photoconversion status
sc.pl.umap(
    adata_cd8,
    color="celltype_coarse",
    show=False, frameon=False, palette=colors50,
    size=1, alpha=0.8,
    save="_mousecd8_umap_celltype.svg",
)
plt.close()

sc.pl.umap(
    adata_cd8,
    color="celltype_coarse",
    show=False, frameon=False, palette=colors50,
    size=1, alpha=0.8,
    legend_loc=None,
    save="_mousecd8_umap_celltype_nolegend.svg",
)
plt.close()

treat_palette = {"Isotype": "#1f77b4", "anti-PDL1": "#ff7f0e"}

sc.pl.umap(
    adata_cd8,
    color="treatment",
    show=False, frameon=False, palette=treat_palette,
    size=1, alpha=0.8,
    save="_mousecd8_umap_treatment.svg",
)
plt.close()

sc.pl.umap(
    adata_cd8,
    color="treatment",
    show=False, frameon=False, palette=treat_palette,
    size=1, alpha=0.8,
    legend_loc=None,
    save="_mousecd8_umap_treatment_nolegend.svg",
)
plt.close()

photo_palette = {"green": "#2ca02c", "red": "#d62728"}

sc.pl.umap(
    adata_cd8,
    color="photoconversion_status",
    show=False, frameon=False, palette=photo_palette,
    size=1, alpha=0.8,
    save="_mousecd8_umap_photoconversion.svg",
)
plt.close()

sc.pl.umap(
    adata_cd8,
    color="photoconversion_status",
    show=False, frameon=False, palette=photo_palette,
    size=1, alpha=0.8,
    legend_loc=None,
    save="_mousecd8_umap_photoconversion_nolegend.svg",
)
plt.close()


In [12]:
adata_cd8.obs['celltype_fine_latest'].value_counts()

celltype_fine_latest
CD8T_mem.1         3108
CD8T_exh_term.5    3073
CD8T_exh_int.2     2004
CD8T_exh_term.3    1909
CD8T_cycling       1456
CD8T_nv            1185
CD8T_mem.2         1027
CD8T_exh_prog       874
CD8T_exh_term.4     778
CD8T_exh_int.1      351
CD8T_mem.3          287
CD8T_exh_term.6      93
Name: count, dtype: int64

In [13]:
# Mouse CD8 UMAP coloured by LAG3 T-cell signature score
LAG3_genes = [
    "Cd27", "Cst7", "Lag3", "Irf1", "Itgal", "Cd8a", "Tap1", "Il2rg", "Tnfsf10",
    "Stat1", "Tnfrsf9", "Spn", "Cxcr3", "Ptpn6", "Tap2", "Itgb2", "Csf1", "Cd3e",
    "Gzma", "Nlrc5", "Rasgrp1", "Gzmk", "Ntng2", "Gbp5", "Pik3cd", "Gbp1", "Jak3",
    "Hnrnpll", "Il2rb", "Igf2r", "Apobec3g", "Gpr174", "Cd2", "Traf5", "Itga4",
    "Sema4d", "Cdh6", "Itgb7", "Cd247", "Cd8b", "Csk", "Tox2", "Grk2", "Prkch",
    "Sirpg", "Apmap", "Apobec3c", "Sh2d2a", "Il16", "Lat", "Il10ra", "Vegfa", "Lbh",
    "Hdac1", "Cd3g", "Tnfrsf14", "Ikzf3", "Cd82", "Cpt1a", "Chd3", "Cp", "Tox",
    "Faslg", "Adar", "Rassf5", "C4b", "Pdia3", "Btn3a1", "Fcrl3", "Mctp2", "Ccr5",
    "Eomes", "Ubd", "Ogt", "Dpp4", "Klrk1", "Tnfrsf1b", "Atxn1", "Lck", "Pecam1",
    "Ubash3a", "Kmt2a", "Ciita", "Ncor2", "Ucp2", "Wars", "Cd38", "Hnrnph1", "Hnrnpd",
    "Rassf1", "Adgre5", "Il21r", "Gtf2i", "Itm2a", "Cd96", "Dnm2", "Havcr2", "Atxn2l",
    "Prf1", "Gusb",
]

sc.tl.score_genes(
    adata_cd8,
    gene_list=LAG3_genes,
    score_name="LAG3_TC_score",
    use_raw=False,
)

values = adata_cd8.obs["LAG3_TC_score"].values
vmin, vmax = np.percentile(values, [20.0, 95])
sc.pl.umap(
    adata_cd8,
    color="LAG3_TC_score",
    show=False, frameon=False, color_map="plasma",
    size=1, alpha=0.8,
    vmin=vmin, vmax=vmax,
    save="_mousecd8_umap_lag3tc_score.svg",
)
plt.close()


computing score 'LAG3_TC_score'
       'Btn3a1', 'Fcrl3'],
      dtype='object')
    finished: added
    'LAG3_TC_score', score of gene set (adata.obs).
    795 total control genes are used. (0:00:00)


In [14]:
# Mean score per condition x cell type
mean_table = (
    adata_cd8.obs
    .groupby(['condition', 'celltype_coarse'])['LAG3_TC_score']
    .mean()
    .unstack()
)
print("Mean score:")
print(mean_table.round(4).to_string())

count_table = (
    adata_cd8.obs
    .groupby(['condition', 'celltype_coarse'])
    .size()
    .unstack(fill_value=0)
)
print("Cell counts:")
print(count_table.to_string())


Mean score:
celltype_coarse  CD8T_cycling  CD8T_exh_int  CD8T_exh_prog  CD8T_exh_term  CD8T_mem  CD8T_nv
condition                                                                                   
Green_Isotype          0.3270        0.3844         0.3315         0.3964    0.3516   0.2154
Green_anti-PDL1        0.3725        0.3989         0.3717         0.4379    0.4042   0.2445
Red_Isotype            0.3266        0.3882         0.3357         0.3927    0.3449   0.2235
Red_anti-PDL1          0.3722        0.4168         0.3926         0.4493    0.3727   0.2328
Cell counts:
celltype_coarse  CD8T_cycling  CD8T_exh_int  CD8T_exh_prog  CD8T_exh_term  CD8T_mem  CD8T_nv
condition                                                                                   
Green_Isotype             260          1378            223            767      2226      815
Green_anti-PDL1            26           238             59            108       792       94
Red_Isotype               762           514  

/tmp/ipykernel_1191634/2742344474.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['condition', 'celltype_coarse'])['LAG3_TC_score']
/tmp/ipykernel_1191634/2742344474.py:13: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(['condition', 'celltype_coarse'])


In [ ]:
# Dotplot of LAG3 TC score split by cell type and treatment
mask = adata_cd8.obs["celltype_coarse"].isin(["CD8T_exh_term", "CD8T_mem"])
adata_sub = adata_cd8[mask].copy()

adata_sub.obs["celltype_x_treatment"] = (
    adata_sub.obs["celltype_coarse"].astype(str) + " | " +
    adata_sub.obs["treatment"].astype(str)
)

score_vals = adata_sub.obs["LAG3_TC_score"].values.reshape(-1, 1)
adata_dot_sub = ad.AnnData(
    X=score_vals,
    obs=adata_sub.obs.copy(),
    var=pd.DataFrame(index=["LAG3 TC score"]),
)

sc.pl.dotplot(
    adata_dot_sub,
    var_names=["LAG3 TC score"],
    groupby="celltype_x_treatment",
    standard_scale="var",
    show=False,
    save="_mousecd8_dotplot_lag3tc_celltype_treatment.svg",
)
plt.close()


In [17]:
# Differential expression between anti-PDL1 and Isotype within CD8T_exh_term
adata_ct = adata_sub[adata_sub.obs["celltype_coarse"] == "CD8T_exh_term"].copy()
sc.tl.rank_genes_groups(
    adata_ct,
    groupby="treatment",
    groups=["anti-PDL1"],
    reference="Isotype",
    method="wilcoxon",
    use_raw=False,
)
sc.pl.rank_genes_groups(
    adata_ct, n_genes=100, sharey=False,
    show=False,
    save="_mousecd8_rankgenes_antiPDL1_vs_isotype.svg",
)
plt.close()

result_df = sc.get.rank_genes_groups_df(adata_ct, group="anti-PDL1")
upregulated = result_df[(result_df["pvals_adj"] < 0.05) & (result_df["logfoldchanges"] > 0)]
print(f"Upregulated genes in anti-PDL1 vs Isotype (CD8T_exh_term): {len(upregulated)}")


ranking genes
    finished: added to `.uns['rank_genes_groups']`
    'names', sorted np.recarray to be indexed by group ids
    'scores', sorted np.recarray to be indexed by group ids
    'logfoldchanges', sorted np.recarray to be indexed by group ids
    'pvals', sorted np.recarray to be indexed by group ids
    'pvals_adj', sorted np.recarray to be indexed by group ids (0:00:02)
Upregulated genes in anti-PDL1 vs Isotype (CD8T_exh_term): 1871


In [18]:
key_genes = ['Cd44', 'Cd28', 'Icos', 'Fasl', 'Ccl5', 'Jak1']
upregulated[upregulated['names'].isin(key_genes)]

,names,scores,logfoldchanges,pvals,pvals_adj
76,Cd44,16.458467,0.619118,7.292325e-61,6.708574e-59
133,Ccl5,14.006749,1.092035,1.417464e-44,7.975509e-43
149,Cd28,13.316867,0.529519,1.846971e-40,9.234353e-39
165,Jak1,12.833849,0.326963,1.059667e-37,4.642098e-36
353,Icos,9.521561,0.362951,1.705970e-21,3.781705e-20
1701,Fasl,2.906672,0.113543,3.652964e-03,1.711719e-02


In [19]:
# Violin plots for key genes with significance annotations
def pval_to_stars(p):
    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"

result_df = sc.get.rank_genes_groups_df(adata_ct, group="anti-PDL1")

fig, axes = plt.subplots(2, 3, figsize=(5, 4))
axes = axes.flatten()
genes = ["Cd44", "Cd28", "Icos", "Fasl", "Ccl5", "Jak1"]

for i, gene in enumerate(genes):
    sc.pl.violin(
        adata_ct,
        keys=gene,
        groupby="treatment",
        stripplot=False,
        jitter=False,
        rotation=90,
        size=1.5,
        scale="width",
        show=False,
        ax=axes[i],
        inner="box",
    )
    axes[i].set_title("")
    axes[i].set_ylabel(gene, fontsize=10)
    axes[i].grid(False)
    axes[i].yaxis.set_major_locator(MultipleLocator(1))
    axes[i].spines["top"].set_visible(False)
    axes[i].spines["right"].set_visible(False)

    pval = result_df.loc[result_df["names"] == gene, "pvals_adj"].values[0]
    stars = pval_to_stars(pval)

    y_range = axes[i].get_ylim()[1] - axes[i].get_ylim()[0]
    y_bar = axes[i].get_ylim()[1] - 0.05 * y_range
    y_tick = 0.02 * y_range
    axes[i].plot([0, 0, 1, 1],
                 [y_bar - y_tick, y_bar, y_bar, y_bar - y_tick],
                 lw=1, color="black")
    axes[i].text(0.5, y_bar + 0.02 * y_range, stars,
                 ha="center", va="bottom", fontsize=9)

    if i < 3:
        axes[i].set_xlabel("")
        axes[i].set_xticklabels([])

plt.tight_layout()
fig.savefig(
    os.path.join(output_dir, "mousecd8_violins_antiPDL1_vs_isotype.svg"),
    bbox_inches="tight",
)
plt.close(fig)


/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: FutureWarning: `scale` is deprecated, use `density_norm` instead
  return fn(*args_all, **kw)
/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: FutureWarning: `scale` is deprecated, use `density_norm` instead
  return fn(*args_all, **kw)
/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: FutureWarning: `scale` is deprecated, use `density_norm` instead
  return fn(*args_all, **kw)
/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: FutureWarning: `scale` is deprecated, use `density_norm` instead
  return fn(*args_all, **kw)
/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: FutureWarning: `scale` is deprecated, use `densit

# T cell - macrophage interaction analysis 

## Ripley H 

In [8]:
def _ripley_H(coords_target: np.ndarray, area: float, r: np.ndarray) -> np.ndarray:
    n = coords_target.shape[0]
    if n < 2 or area <= 0:
        return np.full(r.shape, np.nan, dtype=float)

    d = pdist(coords_target, metric="euclidean")  # length n*(n-1)/2
    counts = (d[None, :] <= r[:, None]).sum(axis=1)  # pairs within r

    intensity = n / area
    K = ((counts * 2) / n) / intensity
    L = np.sqrt(K / np.pi)
    H = L - r
    return H

def ripley_H_per_section_all_types(
    adata,
    cell_type_key: str,
    section_key: str = "section_id",
    spatial_key: str = "spatial",
    max_dist: float = 200.0,
    n_steps: int = 40,
    labels: list | None = None,
    min_points_per_section: int = 10,
) -> pd.DataFrame:
    r = np.linspace(0, max_dist, n_steps)
    rows = []

    if labels is None:
        labels = pd.unique(adata.obs[cell_type_key].dropna())

    for sid in pd.unique(adata.obs[section_key]):
        ad = adata[adata.obs[section_key] == sid]
        coords_all = np.asarray(ad.obsm[spatial_key])

        if coords_all.shape[0] < 3:
            continue

        hull = ConvexHull(coords_all)
        area = float(hull.volume)  # 2D area

        obs_lab = ad.obs[cell_type_key]

        for lab in labels:
            mask = np.asarray(obs_lab == lab)
            if mask.sum() < min_points_per_section:
                continue

            H = _ripley_H(coords_all[mask], area, r)
            rows.append(pd.DataFrame({"section": sid, "label": lab, "r": r, "H": H}))

    if not rows:
        return pd.DataFrame(columns=["section", "label", "r", "H"])
    return pd.concat(rows, ignore_index=True)

def aggregate_curve_mean(
    df: pd.DataFrame,
    x: str = "r",
    y: str = "H",
    group: str = "section",
) -> pd.DataFrame:
    mat = df.pivot_table(index=group, columns=x, values=y, aggfunc="mean")
    xs = mat.columns.to_numpy(float)
    vals = mat.to_numpy(float)
    mean = np.nanmean(vals, axis=0)
    return pd.DataFrame({x: xs, "mean": mean})

def aggregate_all_labels_mean(
    df_all: pd.DataFrame,
    label_col: str = "label",
    group: str = "section",
    min_sections: int = 1,
) -> pd.DataFrame:
    out = []
    for lab, dfl in df_all.groupby(label_col):
        n_sec = dfl[group].nunique()
        if n_sec < min_sections:
            continue
        summ = aggregate_curve_mean(dfl, group=group)
        summ[label_col] = lab
        summ["n_sections"] = n_sec
        out.append(summ)

    if not out:
        return pd.DataFrame(columns=["r", "mean", label_col, "n_sections"])
    return pd.concat(out, ignore_index=True)




### Ripley H Macrophages (supp) Y

In [9]:
adata_TAM.obs['Annotation_TAMs_merged'].unique()


['IL13RA2+', 'IL1B+FCGR2A+', 'FOLR2+', 'IFN+', 'IL1B+FCGBP+', 'SPP1+CCR1+', 'MKI67+', 'SPP1+GRN+']
Categories (8, object): ['IL13RA2+', 'FOLR2+', 'IFN+', 'IL1B+FCGBP+', 'IL1B+FCGR2A+', 'MKI67+', 'SPP1+CCR1+', 'SPP1+GRN+']

In [10]:
# Macrophage Ripley H curves per TAM subtype (with legend)
df_all = ripley_H_per_section_all_types(
    adata_TAM,
    cell_type_key="Annotation_TAMs_merged",
    section_key="batch",
    spatial_key="spatial",
    max_dist=200, n_steps=40, labels=None,
    min_points_per_section=10,
)

summary_all = aggregate_all_labels_mean(
    df_all, label_col="label", group="section", min_sections=1,
)

style_map = {
    "IFN+":           {"alpha": 1.0, "linestyle": "-"},
    "IL1B+FCGR2A+":   {"alpha": 1.0, "linestyle": "-"},
    "SPP1+CCR1+":     {"alpha": 1.0, "linestyle": "-"},
    "SPP1+GRN+":      {"alpha": 1.0, "linestyle": "-"},
    "Ambiguous":      {"alpha": 0.5, "linestyle": "--"},
    "FOLR2+":         {"alpha": 0.5, "linestyle": "--"},
    "IL1B+FCGBP+":    {"alpha": 0.5, "linestyle": "--"},
    "MKI67+":         {"alpha": 0.5, "linestyle": "--"},
}

legend_order = [
    "IFN+", "IL1B+FCGR2A+", "SPP1+CCR1+", "SPP1+GRN+",
    "FOLR2+", "IL1B+FCGBP+", "MKI67+", "Ambiguous",
]

fig, ax = plt.subplots(figsize=(3, 3))
for lab, d in summary_all.groupby("label"):
    d = d.sort_values("r")
    style = style_map.get(lab, {})
    ax.plot(d["r"], d["mean"], label=lab,
            alpha=style.get("alpha", 1.0),
            linestyle=style.get("linestyle", "-"),
            linewidth=1.5)

ax.axhline(0, linestyle="--", color="black", linewidth=1)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlabel("radius (µm)", fontsize=7)
ax.set_ylabel("Ripley H", fontsize=7)
ax.set_title("Macrophage Ripley H", fontsize=7)
ax.tick_params(axis="both", labelsize=7)

handles, labels = ax.get_legend_handles_labels()
handle_dict = dict(zip(labels, handles))
ordered_handles = [handle_dict[l] for l in legend_order if l in handle_dict]
ordered_labels  = [l for l in legend_order if l in handle_dict]

ax.legend(ordered_handles, ordered_labels,
          bbox_to_anchor=(1.05, 1), loc="upper left",
          borderaxespad=0.0, fontsize=8,
          title="Label", title_fontsize=9, frameon=False)

plt.tight_layout()
fig.savefig(
    os.path.join(output_dir, "ripley_h_macrophages.svg"),
    bbox_inches="tight",
)
plt.close(fig)


In [11]:
# Macrophage Ripley H curves (no legend)
fig, ax = plt.subplots(figsize=(3, 3))

for lab, d in summary_all.groupby("label"):
    d = d.sort_values("r")
    style = style_map.get(lab, {})
    ax.plot(
        d["r"], d["mean"], label=lab,
        alpha=style.get("alpha", 1.0),
        linestyle=style.get("linestyle", "-"),
        linewidth=1.5,
    )

ax.axhline(0, linestyle="--", color="black", linewidth=1)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlabel("radius (µm)", fontsize=7)
ax.set_ylabel("Ripley H", fontsize=7)
ax.set_title("Macrophage Ripley H", fontsize=7)
ax.tick_params(axis="both", labelsize=7)

plt.tight_layout()
fig.savefig(
    os.path.join(output_dir, "ripley_h_macrophages_nolegend.svg"),
    bbox_inches="tight",
)
plt.close(fig)


### Ripley H T cells (supp) Y

In [12]:
adata_T


AnnData object with n_obs × n_vars = 351533 × 5001
    obs: 'cell_id', 'x_centroid', 'y_centroid', 'transcript_counts', 'control_probe_counts', 'genomic_control_counts', 'control_codeword_counts', 'unassigned_codeword_counts', 'deprecated_codeword_counts', 'total_counts', 'cell_area', 'nucleus_area', 'nucleus_count', 'segmentation_method', 'batch', 'collection site type', 'donor', 'Xenium barcode', 'drug', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'outlier', '_scvi_batch', '_scvi_labels', 'SCVI_CLUSTERS_KEY', 'low_quality_cell_highlight', 'highlight', 'level_1_cell_type', 'level_2_cell_type', 'level_3_cell_type', 'cell_status', 'celltype_scanvi', 'C_scANVI', 'section', 'concate_key', 'inflow_CT', 'inflow_BatchID', 'sig_withinColor_noRiboMt_CD8plusT_EM', 'sig_withinColor_noRiboMt_CD8plusT_EMRA', 'sig_withinColor_noRiboMt_CD8plusT_EX_CCL4L2', 'sig_withinColor_noRib

In [13]:
adata_T.obs['Annotation_XMic_Tcells'].unique()


['CD4+ CXCR4hi memory-like T cells', 'Activated T cells', 'LAMP1+ perivascular-associated T cells', 'LAG3+ IRF1hi IFN-response T cells', 'CD4+ LGMN+ Treg', 'NDRG1hi CD8+ T cells']
Categories (6, string): [Activated T cells, CD4+ CXCR4hi memory-like T cells, CD4+ LGMN+ Treg, LAG3+ IRF1hi IFN-response T cells, LAMP1+ perivascular-associated T cells, NDRG1hi CD8+ T cells]

In [ ]:
# T cell Ripley H curves per subtype
df_all = ripley_H_per_section_all_types(
    adata_T,
    cell_type_key="Annotation_XMic_Tcells",
    section_key="batch",
    spatial_key="spatial",
    max_dist=200, n_steps=40, labels=None,
    min_points_per_section=10,
)

summary_all = aggregate_all_labels_mean(
    df_all, label_col="label", group="section", min_sections=1,
)

ripley_color_map = {
    "Activated T cells":                        "#d60000",
    "CD4+ CXCR4hi memory-like T cells":         "#8c3bff",
    "CD4+ LGMN+ Treg":                          "#018700",
    "LAG3+ IRF1hi IFN-response T cells":        "#00acc6",
    "LAMP1+ perivascular-associated T cells":   "#97ff00",
    "NDRG1hi CD8+ T cells":                     "#ff7ed1",
}

missing = set(summary_all["label"].unique()) - set(ripley_color_map)
assert not missing, f"No colour assigned for: {missing}"

style_map = {
    "CD4+ LGMN+ Treg":                          {"alpha": 1.0, "linestyle": "-"},
    "CD4+ CXCR4hi memory-like T cells":         {"alpha": 1.0, "linestyle": "-"},
    "LAG3+ IRF1hi IFN-response T cells":        {"alpha": 1.0, "linestyle": "-"},
    "Activated T cells":                        {"alpha": 1.0, "linestyle": "--"},
    "NDRG1hi CD8+ T cells":                     {"alpha": 1.0, "linestyle": "--"},
    "LAMP1+ perivascular-associated T cells":   {"alpha": 1.0, "linestyle": "--"},
}

legend_order = [
    "CD4+ LGMN+ Treg",
    "CD4+ CXCR4hi memory-like T cells",
    "LAG3+ IRF1hi IFN-response T cells",
    "Activated T cells",
    "NDRG1hi CD8+ T cells",
    "LAMP1+ perivascular-associated T cells",
]

fig, ax = plt.subplots(figsize=(3, 3))
for lab, d in summary_all.groupby("label"):
    d = d.sort_values("r")
    style = style_map.get(lab, {})
    ax.plot(d["r"], d["mean"], label=lab,
            color=ripley_color_map[lab],
            alpha=style.get("alpha", 1.0),
            linestyle=style.get("linestyle", "-"),
            linewidth=1.5)

ax.axhline(0, linestyle="--", color="black", linewidth=1)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlabel("radius (µm)", fontsize=7)
ax.set_ylabel("Ripley H", fontsize=7)
ax.set_title("T cell Ripley H", fontsize=7)
ax.tick_params(axis="both", labelsize=7)
ax.yaxis.set_major_locator(MultipleLocator(40))

handles, labels = ax.get_legend_handles_labels()
handle_dict = dict(zip(labels, handles))
ordered_handles = [handle_dict[l] for l in legend_order if l in handle_dict]
ordered_labels  = [l for l in legend_order if l in handle_dict]

ax.legend(ordered_handles, ordered_labels,
          bbox_to_anchor=(1.05, 1), loc="upper left",
          borderaxespad=0.0, fontsize=8,
          title="Label", title_fontsize=9, frameon=False)

fig.savefig(
    os.path.join(output_dir, "ripley_h_tcells.svg"),
    bbox_inches="tight",
)
plt.close(fig)


# T cell Ripley H curves 
fig, ax = plt.subplots(figsize=(3, 3))

for lab, d in summary_all.groupby("label"):
    d = d.sort_values("r")
    style = style_map.get(lab, {})
    ax.plot(
        d["r"], d["mean"], label=lab,
        color=ripley_color_map[lab],
        alpha=style.get("alpha", 1.0),
        linestyle=style.get("linestyle", "-"),
        linewidth=1.5,
    )

ax.axhline(0, linestyle="--", color="black", linewidth=1)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_xlabel("radius (µm)", fontsize=7)
ax.set_ylabel("Ripley H", fontsize=7)
ax.set_title("T cell Ripley H", fontsize=7)
ax.tick_params(axis="both", labelsize=7)
ax.yaxis.set_major_locator(MultipleLocator(40))

fig.savefig(
    os.path.join(output_dir, "ripley_h_tcells_nolegend.svg"),
    bbox_inches="tight",
)
plt.close(fig)


## Nearest neighbor enrichment

In [20]:
cluster_key = "level_4_cell_type_with_Xmic_Tcell_and_XmicXint_merged_TAM_annotations"
library_key = "batch"
spatial_key = "spatial"

adata_full = adata_whole.copy()
adata_full.obs[cluster_key] = adata_full.obs[cluster_key].astype("category")

# Pull T-cell / TAM categories directly from the data rather than hardcoding,
# so this works regardless of which labelling scheme the column was built with.
all_cats = list(adata_full.obs[cluster_key].cat.categories)
T_cell_labels = [c for c in all_cats if str(c).startswith("Tcell_")]
TAM_labels    = [c for c in all_cats if str(c).startswith("TAM_")]

sq.gr.spatial_neighbors(
    adata_full,
    spatial_key=spatial_key, library_key=library_key,
    coord_type="generic", delaunay=True,
)

sq.gr.nhood_enrichment(
    adata_full, cluster_key=cluster_key, library_key=library_key,
    n_perms=1000, seed=0, n_jobs=1, numba_parallel=False,
)

z = adata_full.uns[f"{cluster_key}_nhood_enrichment"]["zscore"]
c = adata_full.uns[f"{cluster_key}_nhood_enrichment"]["count"]

z_df = pd.DataFrame(z, index=all_cats, columns=all_cats)
c_df = pd.DataFrame(c, index=all_cats, columns=all_cats)

z_t_vs_tam = z_df.loc[T_cell_labels, TAM_labels]
c_t_vs_tam = c_df.loc[T_cell_labels, TAM_labels]


Creating graph using `generic` coordinates and `None` transform and `50` libraries.
Adding `adata.obsp['spatial_connectivities']`
       `adata.obsp['spatial_distances']`
       `adata.uns['spatial_neighbors']`
Finish (0:07:36)
Calculating neighborhood enrichment using `1` core(s)


  0%|          | 0/1000 [00:00<?, ?/s]

Adding `adata.uns['level_4_cell_type_with_Xmic_Tcell_and_XmicXint_merged_TAM_annotations_nhood_enrichment']`
Finish (0:04:17)


In [ ]:
# Heatmap: T cell vs TAM neighborhood enrichment z-scores
fig, ax = plt.subplots(figsize=(6, 3))

m = np.nanmax(np.abs(z_t_vs_tam.to_numpy()))

sns.heatmap(
    z_t_vs_tam,
    cmap="RdBu_r",
    center=0,
    vmin=-m,
    vmax=m,
    annot=True,
    fmt=".2f",
    linewidths=1,
    linecolor="none",
    cbar_kws={"label": "Neighborhood enrichment z-score"},
    ax=ax,
)

ax.grid(False)
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1)

ax.collections[0].set_edgecolor("face")

ax.set_xlabel("TAM labels", fontsize=7)
ax.set_ylabel("T cell labels", fontsize=7)
ax.set_title("T cell vs TAM neighborhood enrichment", fontsize=7)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0, ha="right", fontsize=7)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=7)

plt.tight_layout()
fig.savefig(
    os.path.join(output_dir, "nhood_enrichment_tcell_vs_tam_heatmap.svg"),
    bbox_inches="tight",
)
plt.close(fig)


# B cells

### B cell per-section analysis (supp) Y

In [8]:
# B lymphocyte counts inside vs outside TLS crops per section
if "section_name" not in adata_whole.obs.columns:
    adata_whole.obs["section_name"] = (
        adata_whole.obs["batch"].astype(str)
        .map(lambda x: os.path.basename(x).replace(".h5ad", ""))
    )

TLS_crop_coords = {
    "CV6-KID-0-FT-1":    (2500, 4000, 500, 2000),
    "CV9-KID-0-FT-2":    (2400, 3500, 3500, 5200),
    "DI10-KID-0-FT-3":   (2500, 3500, 1500, 2600),
    "DI13-KID-0-FT-1":   (600, 1100, 2300, 2900),
    "CV1-KID-0-FO-1":    (300, 4100, 2000, 3800),
    "CV1-KID-0-FT-2":    (1500, 3500, 1000, 2500),
    "CV5-KID-0-FO-1":    (3900, 5500, 700, 3400),
    "CV7-KID-0-FT-2-s3": (100, 1200, 200, 1700),
}

rows = []
for section, (x1, x2, y1, y2) in TLS_crop_coords.items():
    a_sec = adata_whole[adata_whole.obs["section_name"] == section]
    in_crop = (
        (a_sec.obs["x_centroid"] > x1) & (a_sec.obs["x_centroid"] < x2) &
        (a_sec.obs["y_centroid"] > y1) & (a_sec.obs["y_centroid"] < y2)
    )
    is_b = a_sec.obs["level_3_cell_type"].eq("B lymphocyte")

    rows.append({
        "section": section,
        "Within TLS crop":  int((is_b & in_crop).sum()),
        "Outside TLS crop": int((is_b & ~in_crop).sum()),
    })

plot_df = pd.DataFrame(rows).set_index("section")

ax = plot_df[["Within TLS crop", "Outside TLS crop"]].plot(
    kind="bar", stacked=True, figsize=(6, 3),
    color=["#E74C3C", "#3498DB"],
)
ax.set_ylabel("B lymphocyte count", fontsize=7)
ax.set_xlabel("", fontsize=7)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(fontsize=7)
ax.grid(False)
plt.xticks(rotation=45, ha="right", fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "bcell_tls_counts_per_section.svg"), bbox_inches="tight")
plt.close()


### B cell spatial maps (supp) Y

In [9]:
# Per-section spatial maps highlighting B lymphocytes
target_cell_type = "B lymphocyte"

output_dir_spatial = os.path.join(output_dir, "bcell_spatial_highlight")
os.makedirs(output_dir_spatial, exist_ok=True)

unique_batches = sorted(adata_whole.obs["batch"].unique())
cell_types = sorted(adata_whole.obs["level_3_cell_type"].dropna().unique())

palette = {
    ct: "#d62728" if ct == target_cell_type else "#e0e0e0"
    for ct in cell_types
}

for batch_id in unique_batches:
    fig, ax = plt.subplots(figsize=(6, 6))

    adata_batch = adata_whole[adata_whole.obs["batch"] == batch_id].copy()
    is_target = adata_batch.obs["level_3_cell_type"] == target_cell_type
    adata_batch = adata_batch[np.argsort(is_target.values)]

    sc.pl.spatial(
        adata_batch, color="level_3_cell_type",
        size=1.5, spot_size=20, frameon=False, show=False, ax=ax,
        legend_loc=None, palette=palette,
    )

    raw_name = Path(str(batch_id)).stem
    safe_batch = re.sub(r"[^A-Za-z0-9._-]+", "_", raw_name)
    plt.tight_layout()
    fig.savefig(f"{output_dir_spatial}/{safe_batch}.svg", bbox_inches="tight")
    plt.close(fig)


/tmp/ipykernel_3431721/1411725544.py:22: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/scanpy/plotting/_utils.py:465: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns[f"{value_to_plot}_colors"] = colors_list
/tmp/ipykernel_3431721/1411725544.py:22: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/scanpy/plotting/_utils.py:465: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns[f"{value_to_plot}_colors"] = colors_list
/tmp/ipykernel_3431721/1411725544.py:22: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(
/nfs/team361/zh4/Environments/mintflow_perturbation_2/lib/python3.11/site-packages/scanpy/plotting/_utils.py:465: Imp

### B cell interaction analysis and neighborhood enrichment (supp) Y

In [9]:
# Derive TLS crops directly from adata_whole using the crop coords from
# the sectioning analysis notebook, then overlay level-4 T-cell / TAM
# annotations from adata_T / adata_TAM onto the cropped cells.

# --- build (batch_stem, cell_id) -> label maps from T / TAM adatas -------
def _build_ref_level4(adata_ref, annotation_col):
    ref_df = adata_ref.obs[["batch", annotation_col]].copy()
    ref_df["cell_id"] = adata_ref.obs_names.astype(str)
    ref_df["batch"] = ref_df["batch"].astype(str).map(lambda x: Path(x).stem)
    s = ref_df.set_index(["batch", "cell_id"])[annotation_col]
    n_dup = s.index.duplicated(keep="first").sum()
    if n_dup > 0:
        print(f"Found {n_dup} duplicated (batch, cell_id) entries in {annotation_col}; keeping first.")
    return s[~s.index.duplicated(keep="first")]


ref_level4_t   = _build_ref_level4(adata_T,   "Annotation_XMic_Tcells")
ref_level4_mac = _build_ref_level4(adata_TAM, "Annotation_TAMs_merged")


def _add_level4_cell_types(adata_sample, sample_name=None):
    adata_sample.obs["level_4_cell_type"] = adata_sample.obs["level_3_cell_type"].astype(str)

    batch_stems = adata_sample.obs["batch"].astype(str).map(lambda x: Path(x).stem).to_numpy()
    key = pd.MultiIndex.from_arrays(
        [batch_stems, adata_sample.obs_names.astype(str)],
        names=["batch", "cell_id"],
    )

    mapped_t = ref_level4_t.reindex(key)
    mapped_mask_t = mapped_t.notna().to_numpy()
    adata_sample.obs.loc[mapped_mask_t, "level_4_cell_type"] = mapped_t.to_numpy()[mapped_mask_t]

    mapped_mac = ref_level4_mac.reindex(key)
    mapped_mask_mac = mapped_mac.notna().to_numpy()
    adata_sample.obs.loc[mapped_mask_mac, "level_4_cell_type"] = mapped_mac.to_numpy()[mapped_mask_mac]

    n_overlap = int((mapped_mask_t & mapped_mask_mac).sum())
    if n_overlap:
        print(f"{sample_name}: {n_overlap} cells matched both T-cell and TAM refs; TAM label kept.")
    return {
        "n_T":       int(mapped_mask_t.sum()),
        "n_mac":     int(mapped_mask_mac.sum()),
        "n_overlap": n_overlap,
    }


# --- crop coords (same dict used earlier) ---------------------------------
TLS_crop_coords = {
    "CV6-KID-0-FT-1":    (2500, 4000, 500, 2000),
    "CV9-KID-0-FT-2":    (2400, 3500, 3500, 5200),
    "DI10-KID-0-FT-3":   (2500, 3500, 1500, 2600),
    "DI13-KID-0-FT-1":   (600, 1100, 2300, 2900),
    "CV1-KID-0-FO-1":    (300, 4100, 2000, 3800),
    "CV1-KID-0-FT-2":    (1500, 3500, 1000, 2500),
    "CV5-KID-0-FO-1":    (3900, 5500, 700, 3400),
    "CV7-KID-0-FT-2-s3": (100, 1200, 200, 1700),
}

if "section_name" not in adata_whole.obs.columns:
    adata_whole.obs["section_name"] = (
        adata_whole.obs["batch"].astype(str)
        .map(lambda x: os.path.basename(x).replace(".h5ad", ""))
    )

adatas = []
sample_names = []
mapping_summary = []

for section, (x1, x2, y1, y2) in TLS_crop_coords.items():
    mask_section = adata_whole.obs["section_name"].to_numpy() == section
    a = adata_whole[mask_section].copy()
    if a.n_obs == 0:
        print(f"WARN: no cells found for section {section}")
        continue

    in_crop = (
        (a.obs["x_centroid"] > x1) & (a.obs["x_centroid"] < x2) &
        (a.obs["y_centroid"] > y1) & (a.obs["y_centroid"] < y2)
    ).to_numpy()
    a = a[in_crop].copy()

    # neighbourhood graph on this crop only
    a.uns = {}
    sq.gr.spatial_neighbors(
        a, spatial_key="spatial", library_key=None,
        set_diag=False, delaunay=False, n_neighs=5,
    )

    a.obs["cell_id"] = a.obs_names.astype(str)
    a.obs["source_sample"] = f"{section}_crop"
    stats = _add_level4_cell_types(a, sample_name=section)

    a.obs_names = pd.Index([f"{section}_crop__{cid}" for cid in a.obs["cell_id"].astype(str)])

    adatas.append(a)
    sample_names.append(f"{section}_crop")
    mapping_summary.append({"sample": section, "n_obs": a.n_obs, **stats})

print(pd.DataFrame(mapping_summary))

adata_all = ad.concat(
    adatas,
    join="outer", merge="same",
    label="section_from_file", keys=sample_names,
    index_unique=None, pairwise=True,
)

# Alias for downstream cells
adata_bc = adata_all


Found 3 duplicated (batch, cell_id) entries in Annotation_XMic_Tcells; keeping first.
Creating graph using `generic` coordinates and `None` transform and `1` libraries.
Adding `adata.obsp['spatial_connectivities']`
       `adata.obsp['spatial_distances']`
       `adata.uns['spatial_neighbors']`
Finish (0:00:00)
Creating graph using `generic` coordinates and `None` transform and `1` libraries.
Adding `adata.obsp['spatial_connectivities']`
       `adata.obsp['spatial_distances']`
       `adata.uns['spatial_neighbors']`
Finish (0:00:00)
Creating graph using `generic` coordinates and `None` transform and `1` libraries.
Adding `adata.obsp['spatial_connectivities']`
       `adata.obsp['spatial_distances']`
       `adata.uns['spatial_neighbors']`
Finish (0:00:00)
Creating graph using `generic` coordinates and `None` transform and `1` libraries.
Adding `adata.obsp['spatial_connectivities']`
       `adata.obsp['spatial_distances']`
       `adata.uns['spatial_neighbors']`
Finish (0:00:00)
Creati

In [10]:
adata_all.obs["level_4_cell_type"] = adata_all.obs["level_4_cell_type"].astype("category")

target_a = "B lymphocyte"

sq.gr.spatial_neighbors(
    adata_all,
    coord_type="generic",
    delaunay=True,
    copy=False,
)

# Interaction analysis
im_norm = sq.gr.interaction_matrix(
    adata_all,
    cluster_key="level_4_cell_type",
    normalized=True,
    copy=True,
)

full_cats = adata_all.obs["level_4_cell_type"].cat.categories
im_norm_df = pd.DataFrame(im_norm, index=full_cats, columns=full_cats)

b_profile = (
    im_norm_df.loc[target_a]
    .rename_axis("neighbor_type")
    .reset_index(name="proportion")
    .sort_values("proportion", ascending=False)
    .head(10)
    .copy()
)

b_profile["neighbor_type_plot"] = b_profile["neighbor_type"]


# Neighborhood enrichment 
zscore, count = sq.gr.nhood_enrichment(
    adata_all,
    cluster_key="level_4_cell_type",
    copy=True,
)

cats = adata_all.obs["level_4_cell_type"].cat.categories
z_df = pd.DataFrame(zscore, index=cats, columns=cats)
count_df = pd.DataFrame(count, index=cats, columns=cats)

z_plot = (
    z_df.loc[target_a]
    .rename_axis("neighbor_type")
    .reset_index(name="zscore")
    .sort_values("zscore", ascending=False)
    .head(10)
    .copy()
)

z_plot["neighbor_type_plot"] = z_plot["neighbor_type"]

# B cell interaction profile and neighborhood enrichment bar plots
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=b_profile, x="neighbor_type_plot", y="proportion", ax=ax)

for p in ax.patches:
    h = p.get_height()
    x = p.get_x() + p.get_width() / 2
    ax.annotate(
        f"{h:.3f}",
        (x, h),
        ha="center",
        va="bottom",
        xytext=(0, 3),
        textcoords="offset points",
        fontsize=7,
    )

ax.set_title("B cell interaction profile across cell types (top 10)", fontsize=7)
ax.set_xlabel("")
ax.set_ylabel("Proportion of B cell neighbor edges", fontsize=7)
ax.tick_params(axis="x", rotation=90)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
fig.savefig(
    os.path.join(output_dir, "bcell_interaction_profile_top10.svg"),
    bbox_inches="tight",
)
plt.close(fig)


fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=z_plot, x="neighbor_type_plot", y="zscore", ax=ax)

for p in ax.patches:
    h = p.get_height()
    x = p.get_x() + p.get_width() / 2

    if h >= 0:
        va = "bottom"
        offset = 3
    else:
        va = "top"
        offset = -3

    ax.annotate(
        f"{h:.1f}",
        (x, h),
        ha="center",
        va=va,
        xytext=(0, offset),
        textcoords="offset points",
        fontsize=7,
    )

ax.axhline(0, ls="--", c="black", lw=1)
ax.set_title("Neighborhood enrichment of B cells (top 10)", fontsize=7, pad=10)
ax.set_xlabel("")
ax.set_ylabel("Z-score")
ax.tick_params(axis="x", rotation=90, labelsize=7)
ax.grid(False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_ylim(bottom=-30)

plt.tight_layout()
fig.savefig(
    os.path.join(output_dir, "bcell_nhood_enrichment_top10.svg"),
    bbox_inches="tight",
)
plt.close(fig)


Creating graph using `generic` coordinates and `None` transform and `1` libraries.


Adding `adata.obsp['spatial_connectivities']`
       `adata.obsp['spatial_distances']`
       `adata.uns['spatial_neighbors']`
Finish (0:00:15)
Calculating neighborhood enrichment using `1` core(s)


  0%|          | 0/1000 [00:00<?, ?/s]